# NumPy for Production Machine Learning and AI

**TrajectAI ML and AI Bootcamp, Week 2: Python's Data Toolkit**

NumPy is the memory layer that every other numerical tool sits on top of. Pandas DataFrames wrap NumPy arrays. Scikit-learn takes NumPy arrays in and returns NumPy arrays. PyTorch and TensorFlow tensors share the exact same memory buffer format, so `torch.from_numpy(x)` costs zero copies. Matplotlib plots NumPy arrays. An image decoded by OpenCV or Pillow is a NumPy array. A batch of audio is a NumPy array.

That means: **whatever you cannot express in NumPy, you cannot express efficiently anywhere in the Python data stack.**

### How this notebook is organised

Every section follows the same shape:

1. **What it is** in plain language.
2. **Why production cares**, the failure that happens when you get it wrong.
3. **Where it shows up**, concrete use cases from real ML and AI systems.
4. **Code** you can run and modify.
5. **Pitfall** boxes for the bugs that actually reach production.

You can read it top to bottom as a beginner, or jump to a section as a reference later.

### Roadmap

| # | Section | Core question it answers |
|---|---------|--------------------------|
| 1 | Why NumPy exists | Why not just use Python lists? |
| 2 | Creating arrays | How does data enter NumPy? |
| 3 | Data types and memory | Why is my training job out of memory? |
| 4 | Dimensions and shapes | What is 1D, 2D, 3D, 4D, nD *in production*? |
| 5 | Indexing, slicing, views | Why did my array change when I never touched it? |
| 6 | Boolean masks and fancy indexing | How do I filter, gather, and one-hot? |
| 7 | Axes and reductions | What does `axis=0` actually mean? |
| 8 | Broadcasting | Why did my loss silently become a matrix? |
| 9 | Vectorization | How do I delete my for loops? |
| 10 | Linear algebra | How do neural network layers and attention work? |
| 11 | Numerical stability | Why is my loss NaN? |
| 12 | Randomness and reproducibility | Why can nobody reproduce my results? |
| 13 | Reshaping and combining | How do I collate a batch? |
| 14 | Sliding windows and strides | How do I window time series and patchify images? |
| 15 | Sorting, searching, set ops | How do I get top-k recommendations fast? |
| 16 | Saving, loading, data bigger than RAM | How do I handle a 200 GB dataset? |
| 17 | Performance engineering | Why is my vectorized code still slow? |
| 18 | Testing numerical code | How do I unit test math? |
| 19 | Production pitfalls checklist | What breaks most often? |
| 20 | End to end mini project | Can I build a full pipeline with only NumPy? |

### Setup

In [1]:
pip install numpy

Note: you may need to restart the kernel to use updated packages.


In [2]:
import numpy as np
import sys, time

print("NumPy version:", np.__version__)
print("Python version:", sys.version.split()[0])

# Reproducibility for the whole notebook.
# Modern NumPy style: an explicit Generator object, not the global np.random.seed.
# Section 12 explains why this matters in production.
rng = np.random.default_rng(seed=42)

NumPy version: 1.26.3
Python version: 3.13.5


---
# 1. Why NumPy exists

## What it is

A Python list of numbers is a list of **pointers** to individual Python objects. Each `int` in that list is a full heap object with a reference count, a type pointer, and a value. Iterating it means chasing pointers all over RAM and paying interpreter overhead on every single element.

A NumPy array is **one flat block of raw memory** holding values of one fixed type, plus a small header describing the shape. Operations on it run in compiled C or Fortran loops over that contiguous block, often using SIMD instructions that process 4, 8, or 16 numbers per CPU cycle.

## Why production cares

The gap is typically 10x to 100x in speed and 3x to 10x in memory. At small scale it is invisible. At production scale it decides whether:

- a feature pipeline finishes in the 15 minute batch window or misses the SLA,
- an inference service answers in 40 ms or 900 ms,
- a training epoch takes 6 minutes or 6 hours,
- a dataset fits in a machine's RAM at all.

## Where it shows up

- **Feature engineering at scale.** Computing 200 features over 50 million rows nightly.
- **Real time inference.** Preprocessing a request (tokenize, normalize, reshape) inside a latency budget.
- **Data loaders.** Decoding and augmenting images fast enough to keep a GPU at high utilisation. A slow NumPy path means an idle, expensive GPU.
- **Zero copy handoff.** `torch.from_numpy(arr)` shares memory with the array, so a well shaped NumPy pipeline hands data to the GPU with no serialization cost.

In [3]:
# Speed: Python list vs NumPy array, same computation.
n = 1_000_000
py_list = list(range(n))
np_arr = np.arange(n)

t0 = time.perf_counter()
list_result = [x * 2 + 1 for x in py_list]
t_list = time.perf_counter() - t0

t0 = time.perf_counter()
arr_result = np_arr * 2 + 1
t_numpy = time.perf_counter() - t0

print(f"Python list comprehension : {t_list*1000:8.2f} ms")
print(f"NumPy vectorized          : {t_numpy*1000:8.2f} ms")
print(f"Speedup                   : {t_list/t_numpy:8.1f}x")

Python list comprehension :   136.80 ms
NumPy vectorized          :     7.61 ms
Speedup                   :     18.0x


In [4]:
# Memory: the same one million integers.
list_bytes = sys.getsizeof(py_list) + sum(sys.getsizeof(x) for x in py_list[:1000]) / 1000 * n
arr_bytes = np_arr.nbytes

print(f"Python list  : {list_bytes/1e6:8.1f} MB (approx, pointers plus int objects)")
print(f"NumPy int64  : {arr_bytes/1e6:8.1f} MB")
print(f"NumPy int32  : {np_arr.astype(np.int32).nbytes/1e6:8.1f} MB")
print(f"NumPy float32: {np_arr.astype(np.float32).nbytes/1e6:8.1f} MB")

# Production framing: a 10 million row by 100 feature float32 matrix
rows, feats = 10_000_000, 100
print(f"\n10M x 100 float32 matrix: {rows*feats*4/1e9:.1f} GB")
print(f"10M x 100 float64 matrix: {rows*feats*8/1e9:.1f} GB  <- same data, double the bill")

Python list  :     36.0 MB (approx, pointers plus int objects)
NumPy int64  :      8.0 MB
NumPy int32  :      4.0 MB
NumPy float32:      4.0 MB

10M x 100 float32 matrix: 4.0 GB
10M x 100 float64 matrix: 8.0 GB  <- same data, double the bill


In [5]:
# The array header: shape and dtype describe how to read one flat buffer.
a = np.arange(12, dtype=np.float32).reshape(3, 4)

print("values      :\n", a)
print("shape       :", a.shape)        # logical structure
print("dtype       :", a.dtype)        # element type, fixed for the whole array
print("ndim        :", a.ndim)         # number of axes
print("size        :", a.size)         # total elements
print("itemsize    :", a.itemsize, "bytes per element")
print("nbytes      :", a.nbytes, "bytes total")
print("strides     :", a.strides, "bytes to step along each axis")
print("C contiguous:", a.flags["C_CONTIGUOUS"])

values      :
 [[ 0.  1.  2.  3.]
 [ 4.  5.  6.  7.]
 [ 8.  9. 10. 11.]]
shape       : (3, 4)
dtype       : float32
ndim        : 2
size        : 12
itemsize    : 4 bytes per element
nbytes      : 48 bytes total
strides     : (16, 4) bytes to step along each axis
C contiguous: True


---
# 2. Creating arrays

## What it is

Every array comes from one of four places: a Python sequence, a constructor that fills a shape (`zeros`, `ones`, `full`, `empty`), a range generator (`arange`, `linspace`), or a random generator. In real systems most arrays come from a fifth place, a file or a network payload, which Section 16 covers.

## Why production cares

The constructor you pick sets the dtype, and the dtype sets your memory bill and your numerical behaviour. `np.array([1, 2, 3])` gives int64 on Linux and macOS. `np.zeros(5)` gives float64. Neither is what a GPU wants. Choosing explicitly at creation is cheaper than converting later, because a conversion is a full copy.

## Where it shows up

- `np.zeros` / `np.ones`: preallocating an output buffer you fill in a loop, accumulators for running sums, initial hidden states for an RNN, an all pass attention mask.
- `np.full`: filling a padded sequence tensor with the pad token id, or filling a masked score matrix with `-inf` before a softmax.
- `np.empty`: preallocating a large buffer you will overwrite completely. Faster than `zeros` because it skips zeroing, dangerous if you read before writing.
- `np.arange`: token positions, epoch counters, row indices for shuffling.
- `np.linspace`: learning rate schedules, evaluation thresholds for a precision recall curve, grid points for plotting a decision boundary.
- `np.eye` / `np.identity`: the identity in a ridge regression normal equation, a residual or skip connection as a matrix, an initial state transition matrix in a Kalman filter.

In [6]:
# From Python sequences. NumPy infers dtype, which is often not what you want.
from_list  = np.array([1, 2, 3, 4, 5])
from_tuple = np.array((1.5, 2.5, 3.5))
from_2d    = np.array([[1, 2, 3], [4, 5, 6]])

print(from_list, from_list.dtype)     # int64: 8 bytes per number
print(from_tuple, from_tuple.dtype)   # float64
print(from_2d.shape, from_2d.dtype)

# Production habit: state the dtype you want.
# features = np.array([1, 2, 3, 4, 5], dtype=np.float32)
# labels   = np.array([0, 1, 1, 0, 1], dtype=np.int64)
# print(features.dtype, labels.dtype)

[1 2 3 4 5] int64
[1.5 2.5 3.5] float64
(2, 3) int64


In [7]:
from_2d

array([[1, 2, 3],
       [4, 5, 6]])

In [ ]:
# Shape filling constructors.
print("zeros(3,4) accumulator:\n", np.zeros((3, 4), dtype=np.float32))
print("\nones(2,3) mask:\n", np.ones((2, 3), dtype=bool))
print("\nfull for a padded batch of token ids (pad id = 0):\n",
      np.full((2, 6), 0, dtype=np.int32))

# -inf fill is how causal and padding masks are applied before a softmax:
# masked positions get -inf, so exp(-inf) = 0 and they receive zero attention.
scores = np.full((4, 4), -np.inf, dtype=np.float32)
print("\nmask fill value example:\n", scores[:2])

# empty allocates without initialising. Use only when you overwrite everything.
buf = np.empty((2, 3), dtype=np.float32)
buf[:] = 7.0                      # overwrite before any read
print("\nempty then filled:\n", buf)

In [8]:
# Ranges.
print("arange(10)            :", np.arange(10))
# print("arange(5, 15, 3)      :", np.arange(5, 15, 3))

# # Warning: arange with a float step accumulates rounding error and the element
# # count is not guaranteed. Use linspace when the count matters.
# print("arange(0, 1, 0.1) len :", len(np.arange(0, 1, 0.1)))
# print("linspace(0, 1, 11) len:", len(np.linspace(0, 1, 11)))

# # Production: a linear learning rate warmup then decay schedule.
# warmup = np.linspace(0.0, 1e-3, 5)
# decay  = np.linspace(1e-3, 1e-5, 10)
# lr_schedule = np.concatenate([warmup, decay])
# print("\nlr schedule:", np.round(lr_schedule, 6))

# # Production: thresholds for sweeping a classifier's operating point.
# thresholds = np.linspace(0.0, 1.0, 101)
# print("thresholds:", thresholds[:5], "...", thresholds[-3:])

arange(10)            : [0 1 2 3 4 5 6 7 8 9]


In [12]:
np.linspace(0,10,4)

array([ 0.        ,  3.33333333,  6.66666667, 10.        ])

In [ ]:
# Identity matrices.
I = np.eye(4, dtype=np.float32)
print("identity:\n", I)

# Production: ridge regression closed form, w = (X^T X + lambda I)^-1 X^T y.
# The lambda * I term is what makes the inverse numerically stable.
X = rng.normal(size=(50, 4)).astype(np.float32)
y = rng.normal(size=50).astype(np.float32)
lam = 1.0
w = np.linalg.solve(X.T @ X + lam * np.eye(4, dtype=np.float32), X.T @ y)
print("\nridge weights:", w)

# eye with an offset k builds banded matrices, used for causal masks
# and for finite difference operators.
print("\ncausal shape helper, np.eye(5, k=1):\n", np.eye(5, 5, k=1, dtype=int))

In [ ]:
# Copying the shape of an existing array is the most common production pattern:
# "give me a buffer exactly like this one".
template = np.arange(6, dtype=np.float32).reshape(2, 3)

print("zeros_like :\n", np.zeros_like(template))
print("ones_like  :\n", np.ones_like(template))
print("full_like  :\n", np.full_like(template, np.nan))
# These preserve shape AND dtype, so they never silently upcast to float64.

---
# 3. Data types and memory

## What it is

A NumPy array has exactly one dtype, fixed for every element. The dtype decides three things: how many bytes each element takes, what range of values is representable, and how much precision you get.

## Why production cares

Dtype is the single biggest lever you have over memory, and memory is what decides your batch size, which decides your training throughput and your serving cost.

- float64 to float32 halves memory and roughly doubles throughput on most hardware.
- GPUs are built for float32 and float16. Feeding float64 into a deep learning framework either errors out or silently costs you a conversion copy on every batch.
- int8 quantized weights are 4x smaller than float32, which is how large models fit on small devices.

Getting dtype wrong causes two classic production incidents: an out of memory crash at 3 am, and a **silent integer overflow** that produces plausible looking but wrong numbers with no exception raised.

## Where it shows up

| Data | Right dtype | Reason |
|------|-------------|--------|
| Model weights and activations | `float32` | Framework standard, GPU native |
| Mixed precision training | `float16` / `bfloat16` | Half the memory, tensor core speed |
| Quantized deployed models | `int8` / `uint8` | 4x smaller, integer arithmetic on edge devices |
| Raw images | `uint8` | 0 to 255 is exactly one byte per channel |
| Class labels | `int64` (or `int32`) | Framework loss functions expect integer indices |
| Token ids | `int32` | Vocabularies are under 2 billion |
| Masks and filters | `bool` | One byte per element, and it enables boolean indexing |
| Money and exact counts | `int64` or `decimal` | Never float, floats cannot represent 0.1 exactly |

In [ ]:
# The dtype zoo and what each costs.
for dt in [np.int8, np.int32, np.int64, np.float16, np.float32, np.float64, np.bool_]:
    info = np.iinfo(dt) if np.issubdtype(dt, np.integer) else None
    rng_txt = f"range [{info.min}, {info.max}]" if info else ""
    print(f"{np.dtype(dt).name:10s} {np.dtype(dt).itemsize} bytes  {rng_txt}")

In [ ]:
# The memory conversation you will actually have with your team.
batch, channels, height, width = 64, 3, 224, 224   # one batch of ImageNet sized images
elems = batch * channels * height * width

for dt in [np.uint8, np.float16, np.float32, np.float64]:
    mb = elems * np.dtype(dt).itemsize / 1e6
    print(f"{np.dtype(dt).name:8s}: {mb:8.1f} MB per batch")

print("\nSame pixels. uint8 to float64 is a 8x memory difference.")
print("This is why data loaders keep images as uint8 until the last possible moment,")
print("then convert to float32 on the GPU rather than in RAM.")

In [ ]:
# Silent integer overflow: no exception, just wrong numbers.
small = np.array([120, 125, 127], dtype=np.int8)   # int8 max is 127
print("int8 values      :", small)
print("after adding 10  :", small + np.int8(10), "  <- wrapped around to negative")

# Where this bites: accumulating counts or sums in a narrow integer dtype.
counts = np.array([30000, 30000], dtype=np.int16)  # int16 max is 32767
print("\nint16 sum        :", counts.sum(), "<- overflowed")
print("safe accumulation:", counts.sum(dtype=np.int64))

In [ ]:
# Float precision is finite. This is not a NumPy bug, it is binary floating point.
print("0.1 + 0.2 == 0.3 :", 0.1 + 0.2 == 0.3)
print("difference       :", 0.1 + 0.2 - 0.3)
print("correct check    :", np.isclose(0.1 + 0.2, 0.3))

# float32 has about 7 decimal digits of precision, float64 about 16.
big32 = np.float32(1e8)
print("\nfloat32 1e8 + 1  :", big32 + np.float32(1.0), "<- the 1 disappeared")
print("float64 1e8 + 1  :", np.float64(1e8) + 1.0)

# Production consequence: summing a million small gradients in float32 loses
# accuracy. Frameworks accumulate in float32 even when the data is float16.
x16 = np.full(100_000, 0.01, dtype=np.float16)
print("\nfloat16 sum      :", x16.sum())
print("float32 accumulate:", x16.sum(dtype=np.float32))

In [ ]:
# Casting rules: NumPy upcasts to the wider type in mixed operations.
a32 = np.ones(3, dtype=np.float32)
a64 = np.ones(3, dtype=np.float64)
print("float32 + float64 ->", (a32 + a64).dtype, " <- your float32 pipeline just doubled")

# A Python float literal is float64, but NumPy keeps float32 for array-scalar ops.
print("float32 array * python float ->", (a32 * 2.0).dtype)

# astype ALWAYS copies. In a hot loop that copy is real cost.
img_u8 = rng.integers(0, 256, size=(224, 224, 3), dtype=np.uint8)
img_f32 = img_u8.astype(np.float32) / 255.0
print("\nuint8 bytes  :", img_u8.nbytes)
print("float32 bytes:", img_f32.nbytes, "(4x, and a full copy was made)")

# Cast once at the boundary of your pipeline, not repeatedly inside it.

---
# 4. Dimensions and shapes, from 0D to nD

## What it is

`ndim` is how many axes an array has. `shape` is a tuple giving the length of each axis. `size` is the product of the shape.

An axis is just a direction you can move in. The **last axis is the fastest changing one in memory** for a C ordered array, which is why the feature axis is almost always last: the numbers you multiply together in a dot product sit next to each other in RAM.

## Why production cares

Nearly every bug you will hit in an ML pipeline is a shape bug. Frameworks and libraries agree on conventions, and the conventions are not optional:

- scikit-learn wants `X` as 2D `(n_samples, n_features)` even when there is one feature.
- PyTorch convolutions want `(batch, channels, height, width)`, called NCHW.
- TensorFlow and Keras default to `(batch, height, width, channels)`, called NHWC.
- Recurrent and transformer models want `(batch, time, features)`.
- Loss functions want logits `(batch, num_classes)` and targets `(batch,)`.

Mismatch produces one of two outcomes. The good outcome is a loud exception. The bad outcome is that broadcasting quietly makes the shapes compatible and you train a model on nonsense. Section 8 shows exactly how that happens.

## The mental model

Read a shape right to left. The rightmost axis is the innermost, most granular unit. Each axis to the left is a container holding the thing on its right.

`(32, 3, 224, 224)` reads as: 32 images, each with 3 channels, each channel a 224 row grid, each row 224 pixels wide.

Now walk through each dimensionality with the production meaning.

In [ ]:
def describe(name, a):
    print(f"{name:22s} ndim={a.ndim}  shape={str(a.shape):22s} size={a.size:>10,}  {a.nbytes/1e6:8.3f} MB")

## 4.1 Zero dimensional, a scalar array

`shape=()`, `ndim=0`, exactly one value with no axes.

You rarely build one by hand, but you get them constantly as the **output of a full reduction**: `arr.sum()`, `arr.mean()`, `loss.item()` style values.

### Production use cases

- **The loss value** for a batch. Backpropagation requires a single scalar to differentiate.
- **Any logged metric**: accuracy, AUC, mean latency, gradient norm. Every number on a training dashboard is a 0D value.
- **Hyperparameters** carried as arrays: learning rate, temperature for softmax sampling, epsilon in a normalization layer, the clip threshold for gradient clipping.
- **Early stopping state**: best validation loss so far.

### Pitfall

A 0D array is not a Python float. `float(x)` or `x.item()` converts it. Keeping thousands of 0D arrays alive in a training loop (for example appending `loss` instead of `loss.item()` to a list) keeps the whole computation graph alive in deep learning frameworks and leaks memory.

In [ ]:
loss = np.array(0.6931)
describe("0D loss", loss)
print("value          :", loss)
print("indexing it    :", loss[()])         # the only valid index
print("as python float:", loss.item(), type(loss.item()))

# Reductions produce 0D results.
batch_errors = np.array([0.5, 0.8, 0.3, 1.2], dtype=np.float32)
mean_loss = batch_errors.mean()
print("\nmean loss ndim :", np.ndim(mean_loss), " value:", mean_loss)

# Temperature scaling for sampling, a 0D hyperparameter in action.
logits = np.array([2.0, 1.0, 0.1], dtype=np.float32)
temperature = np.float32(0.7)
scaled = logits / temperature
print("scaled logits  :", scaled)

## 4.2 One dimensional, a vector

`shape=(n,)`. A flat sequence of numbers. The workhorse shape.

### Production use cases

- **One sample's feature vector.** A single credit application as 40 numbers, a single sensor reading, a single row of a feature store.
- **An embedding.** A word, user, product, or image compressed into 128, 384, 768, or 1536 numbers. Semantic search, recommendation, and retrieval augmented generation all live on 1D embeddings.
- **A time series.** Hourly CPU load, daily revenue, an ECG trace, a mono audio waveform (16000 numbers per second at 16 kHz).
- **Token ids.** One tokenized prompt is a 1D int array of vocabulary indices.
- **Class probabilities.** The softmax output for one sample over `num_classes`.
- **Labels for a batch.** `y_true` and `y_pred` are 1D when you compute accuracy or F1.
- **Model weights of a single linear unit**, or a bias vector, or a per feature mean and standard deviation computed by a scaler.

### Pitfall

`(n,)` and `(n, 1)` and `(1, n)` are three different shapes and they broadcast differently. Scikit-learn raising "Expected 2D array, got 1D array instead" is this exact issue. Fix with `x.reshape(-1, 1)` for a single feature or `x.reshape(1, -1)` for a single sample.

In [ ]:
# A single sample's features.
one_sample = np.array([35.0, 72000.0, 0.31, 7.0], dtype=np.float32)  # age, income, utilization, tenure
describe("1D sample features", one_sample)

# A sentence embedding from a modern text encoder.
embedding = rng.normal(size=384).astype(np.float32)
describe("1D embedding", embedding)

# Ten minutes of 16 kHz mono audio.
audio = rng.normal(size=16_000 * 60 * 10).astype(np.float32)
describe("1D audio 10 min", audio)

# A tokenized prompt.
token_ids = np.array([101, 7592, 2088, 999, 102], dtype=np.int32)
describe("1D token ids", token_ids)

# Softmax output for one sample over 5 classes.
probs = np.array([0.02, 0.71, 0.15, 0.09, 0.03], dtype=np.float32)
describe("1D class probs", probs)
print("\npredicted class:", probs.argmax(), " confidence:", probs.max())

In [ ]:
# The (n,) vs (n,1) distinction that breaks scikit-learn calls.
x = np.array([1.0, 2.0, 3.0, 4.0])
print("x           shape:", x.shape,               "  1D, n samples OR n features, ambiguous")
print("x.reshape(-1,1)  :", x.reshape(-1, 1).shape, " 4 samples, 1 feature   <- single feature model")
print("x.reshape(1,-1)  :", x.reshape(1, -1).shape, " 1 sample,  4 features  <- single prediction")

# Predicting for one sample at inference time is the classic case.
model_input = one_sample.reshape(1, -1)
print("\ninference input shape:", model_input.shape, "(batch of 1)")

In [18]:
zero_d_1= np.array(10)

In [19]:
zero_d_1.shape

()

In [20]:
zero_d_1.ndim

0

In [21]:
zero_d_1.size

1

In [22]:
zero_d_1

array(10)

In [23]:
zero_d_2= np.array(20)

In [24]:
zero_d_2.shape

()

In [25]:
one_d=np.array([zero_d_1,zero_d_2])

In [26]:
one_d

array([10, 20])

In [28]:
one_d.shape # it has 2 zero -d arrays inside

(2,)

In [41]:
one_d

array([10, 20])

In [42]:
one_d[1]

20

## 4.3 Two dimensional, a matrix

`shape=(rows, cols)`. The single most common shape in classical machine learning.

### Production use cases

- **The design matrix** `X` of shape `(n_samples, n_features)`. Every tabular model, every scikit-learn estimator, every CSV loaded through pandas and handed to a model.
- **A batch of embeddings** `(batch_size, embedding_dim)`. What a vector database returns for a multi query search, and what you feed to a similarity computation.
- **A weight matrix** `(in_features, out_features)` in a dense or linear layer. The entire forward pass is `X @ W + b`.
- **A grayscale image** `(height, width)`. An MNIST digit is `(28, 28)`. A medical X ray slice is `(512, 512)`.
- **A confusion matrix** `(num_classes, num_classes)` for evaluation.
- **An attention score matrix** `(seq_len, seq_len)`, where entry `(i, j)` is how much token `i` attends to token `j`.
- **A similarity matrix** `(n_queries, n_documents)` in retrieval and recommendation.
- **A user item interaction matrix** `(n_users, n_items)` in collaborative filtering.
- **A spectrogram** `(n_mels, n_frames)`, audio converted into an image like representation.

### Pitfall

Row major versus column major confusion. In `X`, axis 0 is samples and axis 1 is features. `X.mean(axis=0)` gives the mean **per feature** (what a scaler needs). `X.mean(axis=1)` gives the mean **per sample** (almost never what you want). Section 7 makes this concrete.

In [ ]:
# Design matrix: 1000 customers, 12 features.
X = rng.normal(size=(1000, 12)).astype(np.float32)
describe("2D design matrix", X)
print("per feature mean shape:", X.mean(axis=0).shape, " <- what StandardScaler stores")
print("per sample  mean shape:", X.mean(axis=1).shape, " <- rarely meaningful")

# Batch of sentence embeddings returned by a vector search.
emb_batch = rng.normal(size=(64, 384)).astype(np.float32)
describe("2D embedding batch", emb_batch)

# Dense layer weights: 384 in, 128 out.
W = rng.normal(size=(384, 128)).astype(np.float32) * 0.02
b = np.zeros(128, dtype=np.float32)
hidden = emb_batch @ W + b
describe("2D layer output", hidden)

# A grayscale image.
mnist_digit = rng.integers(0, 256, size=(28, 28), dtype=np.uint8)
describe("2D grayscale image", mnist_digit)

# Attention scores for one sequence of 128 tokens.
attn = rng.normal(size=(128, 128)).astype(np.float32)
describe("2D attention matrix", attn)

# Confusion matrix for a 5 class problem.
cm = rng.integers(0, 50, size=(5, 5))
describe("2D confusion matrix", cm)

In [29]:
two_d=np.array([[1,2,3],
                  [5,6,7]])

In [31]:
two_d.shape # no of rows , no of columns (no of 1-d vectors in row wise, no of 1-d vectors in column wise)

(2, 3)

In [50]:
two_d[0,2],two_d[1,1]

(3, 6)

In [32]:
two_d.size

6

In [33]:
two_d.ndim

2

## 4.4 Three dimensional

`shape=(a, b, c)`. Three different production meanings, and knowing which one you are holding is the whole game.

### Meaning 1: a batch of vectors over time, `(batch, timesteps, features)`

The standard input to LSTMs, GRUs, temporal convolutions, and transformers.

- **Sensor and IoT forecasting**: 32 machines, 48 hourly readings each, 10 sensors per reading gives `(32, 48, 10)`.
- **Financial modelling**: 128 tickers, 60 trading days, 5 values each (open, high, low, close, volume) gives `(128, 60, 5)`.
- **Language models**: 8 sequences, 512 tokens, 768 dimensional embeddings gives `(8, 512, 768)`. This is the shape flowing between every transformer block.
- **Clickstream and session modelling**: users, events in the session, event features.

### Meaning 2: a single colour image, `(height, width, channels)`

- An RGB photo is `(1080, 1920, 3)`. Pillow, OpenCV, and matplotlib all use this layout.
- A satellite image can be `(512, 512, 13)` because it has 13 spectral bands, not just 3.
- A segmentation mask with one plane per class is `(H, W, num_classes)`.

### Meaning 3: a batch of grayscale images or a volume, `(batch, height, width)`

- 64 MNIST digits as `(64, 28, 28)`.
- A stack of medical slices forming a 3D scan.

### Pitfall

The `(H, W, C)` versus `(C, H, W)` confusion. OpenCV hands you `(H, W, C)`. PyTorch wants `(C, H, W)`. Feeding the wrong one usually does not crash, because 3 is a valid axis length either way, it just produces garbage predictions. Convert explicitly with `np.transpose` or `np.moveaxis`.

In [ ]:
# Meaning 1: sequences.
sensor_batch = rng.normal(size=(32, 48, 10)).astype(np.float32)   # machines, hours, sensors
describe("3D sensor batch", sensor_batch)
print("  reads as: 32 machines, 48 hourly steps, 10 sensors each")

llm_hidden = rng.normal(size=(8, 512, 768)).astype(np.float32)    # batch, tokens, model dim
describe("3D transformer state", llm_hidden)
print("  reads as: 8 sequences, 512 tokens, 768 dim embedding per token")
print("  memory for ONE such tensor:", f"{llm_hidden.nbytes/1e6:.1f} MB.",
      "A 32 layer model holds dozens of these.")

# Meaning 2: one colour image.
photo = rng.integers(0, 256, size=(1080, 1920, 3), dtype=np.uint8)
describe("3D RGB photo", photo)
print("  channel 0 (red) shape:", photo[:, :, 0].shape)

# Meaning 3: batch of grayscale.
mnist_batch = rng.integers(0, 256, size=(64, 28, 28), dtype=np.uint8)
describe("3D grayscale batch", mnist_batch)

In [ ]:
# The HWC to CHW conversion every computer vision pipeline performs.
img_hwc = rng.integers(0, 256, size=(224, 224, 3), dtype=np.uint8)   # OpenCV / Pillow layout
img_chw = np.transpose(img_hwc, (2, 0, 1))                          # PyTorch layout
print("HWC (OpenCV, TensorFlow):", img_hwc.shape)
print("CHW (PyTorch)           :", img_chw.shape)
print("same data, no copy? ", np.shares_memory(img_hwc, img_chw), "(it is a view, only strides changed)")

# moveaxis is more readable when you only care about one axis.
print("moveaxis result         :", np.moveaxis(img_hwc, -1, 0).shape)

In [34]:
three_d=np.array(
   [ [[1,2,3],
    [5,6,7]]
    ,
    [[1,2,3],
    [5,6,7]] ]
)

In [36]:
three_d.shape # layers, rows, columns

(2, 2, 3)

In [52]:
three_d[0,1,1]

6

In [37]:
three_d.size

12

## 4.5 Four dimensional

`shape=(a, b, c, d)`. This is the native language of computer vision and attention.

### Meaning 1: a batch of colour images

- **NCHW** `(batch, channels, height, width)`: PyTorch, and what a GPU convolution kernel prefers. A typical training batch is `(32, 3, 224, 224)`.
- **NHWC** `(batch, height, width, channels)`: TensorFlow and Keras default, and what mobile and TPU hardware often prefers.

### Meaning 2: convolution kernel weights

`(out_channels, in_channels, kernel_h, kernel_w)`. A first ResNet layer is `(64, 3, 7, 7)`, meaning 64 filters, each looking at 3 input channels through a 7 by 7 window.

### Meaning 3: multi head attention

`(batch, num_heads, seq_len, head_dim)`. The whole point of splitting into heads is this extra axis, so each head learns a different relationship pattern in parallel. GPT style models spend most of their compute on tensors shaped exactly like this.

### Meaning 4: video, or a batch of volumes

- A short clip is `(frames, height, width, channels)`.
- A batch of 3D medical volumes without a channel axis is `(batch, depth, height, width)`.

### Production framing

Batch size selection is a 4D memory calculation. Doubling batch size doubles activation memory for every layer, which is usually what makes training crash with out of memory.

In [ ]:
# NCHW training batch, and its real memory cost.
batch_nchw = np.zeros((32, 3, 224, 224), dtype=np.float32)
describe("4D batch NCHW", batch_nchw)

batch_nhwc = np.transpose(batch_nchw, (0, 2, 3, 1))
describe("4D batch NHWC", batch_nhwc)

# Convolution weights.
conv1_w = rng.normal(size=(64, 3, 7, 7)).astype(np.float32) * 0.01
describe("4D conv kernel", conv1_w)
print("  64 filters, each 3 channels deep, each 7x7 window")

# Multi head attention state: 8 sequences, 12 heads, 512 tokens, 64 dims per head.
attn_state = np.zeros((8, 12, 512, 64), dtype=np.float32)
describe("4D attention heads", attn_state)
print("  note 12 * 64 = 768, the model dim split across heads")

# The attention score tensor is the memory hog: it is quadratic in sequence length.
scores = np.zeros((8, 12, 512, 512), dtype=np.float32)
describe("4D attention scores", scores)
print("  double the sequence length to 1024 and this becomes",
      f"{8*12*1024*1024*4/1e6:.0f} MB. Quadratic growth is why long context is hard.")

In [ ]:
# Batch size versus memory, the calculation you do before every training run.
def activation_mb(batch, channels=64, hw=112, dtype=np.float32):
    return batch * channels * hw * hw * np.dtype(dtype).itemsize / 1e6

for bs in [16, 32, 64, 128, 256]:
    print(f"batch {bs:4d}: {activation_mb(bs):8.1f} MB for ONE feature map layer")
print("\nMultiply by the number of layers, then by 2 or 3 for gradients and")
print("optimizer state. That is your out of memory error explained.")

In [38]:
four_d=np.array(
  [ [ [[1,2,3],
    [5,6,7]]
    ,
    [[1,2,3],
    [5,6,7]] ]
    ,
    [ [[1,2,3],
    [5,6,7]]
    ,
    [[1,2,3],
    [5,6,7]] ] ]
)

In [53]:
four_d

array([[[[1, 2, 3],
         [5, 6, 7]],

        [[1, 2, 3],
         [5, 6, 7]]],


       [[[1, 2, 3],
         [5, 6, 7]],

        [[1, 2, 3],
         [5, 6, 7]]]])

In [54]:
four_d[0,0,1,2]

7

In [39]:
four_d.shape # how many 3-d we have, how many layers each 3-d have, how many rows each layer have , how many columns each layer have

# (2,2,2,3 )

(2, 2, 2, 3)

In [40]:
four_d.size

24

## 4.6 Five dimensional and beyond

`ndim >= 5` is not exotic, it appears whenever you add a batch axis to something that was already 4D.

### Production use cases

- **Batched video** `(batch, frames, channels, height, width)`, for action recognition or video captioning. Example: `(8, 16, 3, 224, 224)`, 8 clips of 16 frames each.
- **3D medical imaging** `(batch, channels, depth, height, width)`. A batch of CT or MRI volumes for a 3D U-Net segmenting organs or tumours. Example: `(4, 1, 64, 128, 128)`.
- **Mixture of experts weights** `(num_experts, num_layers, in_dim, out_dim, ...)`, held as one tensor so expert selection is a single indexing operation.
- **Ensembles and hyperparameter sweeps** `(n_seeds, n_configs, n_epochs, n_folds, n_metrics)`. Holding an entire experiment grid in one array means you can answer "mean validation AUC per configuration, averaged over seeds and folds" with a single `mean(axis=(0, 3))`.
- **Patch extraction intermediates.** Cutting an image into patches for a Vision Transformer produces a 6D intermediate `(batch, patches_y, patches_x, channels, patch_h, patch_w)` before you flatten it back down. Section 14 builds this.
- **Physics, climate, and simulation grids**: `(time, altitude, latitude, longitude, variable)` in weather models fed to ML surrogates.

### The rule that scales to any n

You never need to visualise 6 dimensions. You need to know, for each axis position, what that axis indexes. Name your axes in a comment, keep the batch axis first, keep the feature or channel axis where your framework expects it, and use `einsum` (Section 10) when the index bookkeeping gets hard.

In [ ]:
# Batched video for action recognition.
video_batch = np.zeros((8, 16, 3, 224, 224), dtype=np.float32)
describe("5D video batch", video_batch)
print("  8 clips, 16 frames each, 3 channels, 224x224")

# Batched 3D medical volumes for segmentation.
ct_batch = np.zeros((4, 1, 64, 128, 128), dtype=np.float32)
describe("5D CT volumes", ct_batch)
print("  4 patients, 1 channel, 64 axial slices of 128x128")

# An experiment grid. This is how you keep a sweep tidy.
seeds, configs, epochs, folds, metrics = 5, 8, 30, 5, 4
grid = rng.random((seeds, configs, epochs, folds, metrics)).astype(np.float32)
describe("5D experiment grid", grid)

# Question: best configuration by final epoch validation metric 0,
# averaged over seeds and folds. One line, no loops.
final_epoch = grid[:, :, -1, :, 0]          # (seeds, configs, folds)
per_config  = final_epoch.mean(axis=(0, 2)) # average out seeds and folds
print("\nmean final metric per config:", np.round(per_config, 4))
print("best config index           :", per_config.argmax())

In [ ]:
# Reading any shape: walk it right to left.
shapes = {
    "()":                    "a single number, a loss or a metric",
    "(768,)":                "one embedding vector",
    "(32, 768)":             "32 embeddings, one per sample in the batch",
    "(32, 512, 768)":        "32 sequences, 512 tokens each, 768 dims per token",
    "(32, 3, 224, 224)":     "32 images, 3 channels, 224 rows, 224 columns",
    "(32, 12, 512, 64)":     "32 sequences, 12 attention heads, 512 tokens, 64 dims per head",
    "(8, 16, 3, 224, 224)":  "8 clips, 16 frames, 3 channels, 224x224 pixels",
}
for s, meaning in shapes.items():
    print(f"{s:24s} {meaning}")

---
# 5. Indexing, slicing, and the view versus copy trap

## What it is

Indexing pulls out elements. Slicing pulls out ranges with `start:stop:step`, where `stop` is excluded. With multiple axes you separate them with commas: `arr[rows, cols]`, `arr[batch, channel, row, col]`.

The critical fact: **a basic slice returns a view, not a copy.** A view is a new header pointing at the same memory. Writing to the view writes to the original.

## Why production cares

This is a genuine source of production incidents. A preprocessing function takes a slice of a shared buffer, normalizes it in place, and silently corrupts the caller's data. In a data loader with reused buffers, you get corrupted training batches that show up as a mysteriously unstable loss curve, days later.

The other half of the fact is a performance win: because slicing is free, cropping an image, taking the last `k` timesteps, or selecting a channel costs nothing. Only copy when you must.

## Where it shows up

- **Cropping and region of interest** extraction in vision pipelines.
- **Windowing time series**: `series[-lookback:]` is the model input at inference.
- **Train, validation, test splits** on already shuffled data: `X[:n_train]`, `X[n_train:n_val]`.
- **Channel selection**: taking the near infrared band out of a satellite image.
- **Causal masking**: `scores[i, i+1:] = -inf` for autoregressive models.
- **Downsampling**: `signal[::4]` to drop the sample rate by 4.
- **Reversing**: `arr[::-1]` for flipping an image or reversing a sequence.

In [ ]:
a = np.arange(10)
print("a               :", a)
print("a[0], a[-1]     :", a[0], a[-1])
print("a[2:7]          :", a[2:7],   " stop is excluded")
print("a[:3]           :", a[:3])
print("a[7:]           :", a[7:])
print("a[::2]          :", a[::2],   " every second element, downsampling")
print("a[::-1]         :", a[::-1],  " reversed, no copy")
print("a[1:8:3]        :", a[1:8:3], " start:stop:step")

In [ ]:
# Multi dimensional indexing.
A = np.arange(24).reshape(2, 3, 4)   # (layers, rows, cols)
print("A shape:", A.shape)
print(A)

print("\nA[0]        ->", A[0].shape,        " first layer, a 2D matrix")
print("A[0, 1]     ->", A[0, 1].shape,     " first layer second row, a 1D vector")
print("A[0, 1, 2]  ->", A[0, 1, 2],        " a single element (0D)")
print("A[:, :, 0]  ->", A[:, :, 0].shape,  " column 0 of every layer")
print("A[..., 0]   ->", A[..., 0].shape,   " same thing, Ellipsis fills the middle axes")
print("A[:, 1:, :2]->", A[:, 1:, :2].shape)

In [ ]:
# Ellipsis and newaxis, the two indexing tools people forget.
batch = rng.normal(size=(8, 3, 32, 32)).astype(np.float32)

# "the last axis, whatever the rank" without hardcoding the number of axes
print("batch[..., 0].shape :", batch[..., 0].shape)

# np.newaxis (an alias for None) inserts an axis of length 1.
v = np.array([1.0, 2.0, 3.0])
print("\nv              :", v.shape)
print("v[:, np.newaxis]:", v[:, np.newaxis].shape, " column vector")
print("v[np.newaxis, :]:", v[np.newaxis, :].shape, " row vector")

# Production: adding a batch axis for a single sample at inference time.
single_image = rng.normal(size=(3, 224, 224)).astype(np.float32)
model_input = single_image[np.newaxis, ...]
print("\nsingle image  :", single_image.shape)
print("as batch of 1 :", model_input.shape)

In [ ]:
# THE VIEW TRAP. Read this cell twice.
original = np.arange(10)
window = original[2:6]          # basic slice -> a VIEW
window[:] = 0                   # writing to the view...

print("window   :", window)
print("original :", original, "  <- the original was modified")
print("shares memory:", np.shares_memory(original, window))
print("window.base is original:", window.base is original)

In [ ]:
# The safe pattern: copy when you intend to own the data.
original = np.arange(10)
window = original[2:6].copy()
window[:] = 0
print("window   :", window)
print("original :", original, "  <- untouched")
print("shares memory:", np.shares_memory(original, window))

# Production rule of thumb:
#   Function that MUTATES an argument -> document it loudly, or copy first.
#   Function that RETURNS a slice     -> the caller now shares your memory.
def normalize_inplace(x):
    # Mutates x. Caller beware.
    x -= x.mean()
    x /= (x.std() + 1e-8)
    return x

def normalize_safe(x):
    # Returns a new array, never touches the input.
    x = np.asarray(x, dtype=np.float32)
    return (x - x.mean()) / (x.std() + 1e-8)

data = np.array([1.0, 2.0, 3.0, 4.0], dtype=np.float32)
out = normalize_safe(data)
print("\nafter normalize_safe, data is still:", data)

In [ ]:
# Production slicing patterns you will write again and again.

# 1. Train / validation / test split on pre-shuffled data.
n = 1000
X_all = rng.normal(size=(n, 8)).astype(np.float32)
y_all = rng.integers(0, 2, size=n)
i_tr, i_va = int(0.7 * n), int(0.85 * n)
X_train, X_val, X_test = X_all[:i_tr], X_all[i_tr:i_va], X_all[i_va:]
print("split sizes:", X_train.shape[0], X_val.shape[0], X_test.shape[0])

# 2. Last k timesteps as the model input at serving time.
series = rng.normal(size=(500,)).astype(np.float32)
lookback = 48
model_input = series[-lookback:]
print("inference window:", model_input.shape)

# 3. Centre crop of an image.
img = rng.integers(0, 256, size=(256, 256, 3), dtype=np.uint8)
crop = 224
top = left = (256 - crop) // 2
cropped = img[top:top + crop, left:left + crop, :]
print("centre crop     :", cropped.shape, "copy made?", not np.shares_memory(img, cropped))

# 4. Selecting spectral bands from satellite imagery.
sat = rng.random((512, 512, 13)).astype(np.float32)
rgb = sat[..., [3, 2, 1]]     # fancy indexing on the last axis -> this DOES copy
nir = sat[..., 7]
print("rgb composite   :", rgb.shape, " nir band:", nir.shape)

# 5. Causal mask for autoregressive attention: no token sees the future.
seq = 6
mask = np.zeros((seq, seq), dtype=np.float32)
for i in range(seq):
    mask[i, i + 1:] = -np.inf
print("\ncausal mask (0 allowed, -inf blocked):\n", mask)
# Vectorized equivalent, no loop:
mask_vec = np.triu(np.full((seq, seq), -np.inf, dtype=np.float32), k=1)
print("matches vectorized version:", np.array_equal(np.nan_to_num(mask, neginf=-1e9),
                                                    np.nan_to_num(mask_vec, neginf=-1e9)))

---
# 6. Boolean masks and fancy indexing

## What it is

Two forms of advanced indexing:

- **Boolean masking**: index with a boolean array of the same shape. You get back the elements where the mask is `True`, always as a 1D array, always a **copy**.
- **Fancy (integer) indexing**: index with an array of positions. You get elements in the order you asked for, with the shape of the index array. Also always a **copy**.

Both are copies, unlike basic slicing. That is the memory tradeoff for the flexibility.

## Why production cares

This is how data cleaning, filtering, class balancing, and embedding lookup are expressed without loops. An embedding lookup in particular, `embedding_table[token_ids]`, is fancy indexing, and it is the first operation in every language model.

## Where it shows up

- **Removing invalid rows**: NaN, negative prices, timestamps in the future, sensors reading zero.
- **Outlier handling**: keeping rows within 3 standard deviations, or clipping at percentiles.
- **Class balancing**: taking all the minority class rows and a random sample of the majority.
- **Segment analysis**: metrics for one country, one device type, one cohort.
- **Embedding lookup**: `E[token_ids]` turns ids into vectors.
- **Gathering top-k results** after a similarity search.
- **Scatter add** for building histograms, confusion matrices, and sparse gradient accumulation.
- **One hot encoding** of labels.
- **Hard example mining**: selecting the highest loss samples for the next batch.

In [ ]:
# Boolean masking basics.
prices = np.array([120.0, -5.0, 340.0, np.nan, 89.0, 15000.0, 42.0], dtype=np.float32)

is_valid = np.isfinite(prices) & (prices > 0)
print("mask       :", is_valid)
print("valid rows :", prices[is_valid])
print("count      :", is_valid.sum(), "of", is_valid.size)

# Combining conditions: use & | ~ with parentheses, NOT and/or/not.
# Python's `and` calls bool() on the array and raises ValueError.
mid = prices[(prices > 50) & (prices < 1000)]
print("mid range  :", mid)

In [ ]:
# Cleaning a real feature matrix: drop rows containing any NaN.
Xd = rng.normal(size=(10, 4)).astype(np.float32)
Xd[2, 1] = np.nan
Xd[7, 3] = np.nan
yd = rng.integers(0, 2, size=10)

row_ok = ~np.isnan(Xd).any(axis=1)     # any NaN across the feature axis
print("rows kept:", row_ok.sum(), "of", len(row_ok))
X_clean, y_clean = Xd[row_ok], yd[row_ok]
print("shapes   :", X_clean.shape, y_clean.shape)

# Alternative in production: impute rather than drop, so you keep the sample.
col_means = np.nanmean(Xd, axis=0)
X_imputed = np.where(np.isnan(Xd), col_means, Xd)
print("any NaN left:", np.isnan(X_imputed).any())

In [ ]:
# Outlier removal with an interquartile range rule, per feature.
Xo = rng.normal(size=(1000, 3)).astype(np.float32)
Xo[::100] *= 50                                  # inject outliers

q1, q3 = np.percentile(Xo, [25, 75], axis=0)
iqr = q3 - q1
lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
inlier = ((Xo >= lo) & (Xo <= hi)).all(axis=1)
print("inliers kept:", inlier.sum(), "of", len(inlier))

# Often better than dropping: winsorize (clip) so you keep the sample count.
X_clipped = np.clip(Xo, lo, hi)
print("max before clip:", Xo.max().round(2), " after:", X_clipped.max().round(2))

In [ ]:
# Fancy indexing: embedding lookup, the first layer of every language model.
vocab_size, embed_dim = 10_000, 64
embedding_table = rng.normal(size=(vocab_size, embed_dim)).astype(np.float32) * 0.02

token_ids = np.array([[101, 2054, 2003, 102],
                      [101, 7592, 999, 102]], dtype=np.int32)   # (batch, seq)

token_vectors = embedding_table[token_ids]                      # fancy indexing
print("token ids     :", token_ids.shape)
print("token vectors :", token_vectors.shape, " (batch, seq, embed_dim)")
print("rule: the result takes the shape of the INDEX array,")
print("      with the indexed axis replaced by the remaining axes of the source.")

In [ ]:
# One hot encoding, two ways.
labels = np.array([2, 0, 1, 2, 1], dtype=np.int64)
num_classes = 3

# Identity trick: clean and fast.
onehot = np.eye(num_classes, dtype=np.float32)[labels]
print("one hot:\n", onehot)

# Explicit version, useful when you need label smoothing.
smooth = 0.1
soft = np.full((len(labels), num_classes), smooth / num_classes, dtype=np.float32)
soft[np.arange(len(labels)), labels] += 1.0 - smooth
print("\nlabel smoothed targets:\n", np.round(soft, 3))
print("rows sum to 1:", np.allclose(soft.sum(axis=1), 1.0))

In [ ]:
# Scatter add with np.add.at: accumulate into repeated indices.
# Plain fancy assignment does NOT accumulate, it keeps only the last write.
counts_wrong = np.zeros(5, dtype=np.int64)
idx = np.array([1, 1, 1, 3, 3])
counts_wrong[idx] += 1                       # only counts each index ONCE
print("wrong (fancy +=):", counts_wrong)

counts_right = np.zeros(5, dtype=np.int64)
np.add.at(counts_right, idx, 1)              # true scatter add
print("right (np.add.at):", counts_right)
print("fastest for 1D counts:", np.bincount(idx, minlength=5))

# Production: building a confusion matrix with one scatter add.
y_true = rng.integers(0, 4, size=200)
y_pred = rng.integers(0, 4, size=200)
cm = np.zeros((4, 4), dtype=np.int64)
np.add.at(cm, (y_true, y_pred), 1)
print("\nconfusion matrix:\n", cm)
print("accuracy:", np.trace(cm) / cm.sum())

In [ ]:
# Class balancing for an imbalanced fraud dataset.
y_imb = np.concatenate([np.zeros(9500, dtype=np.int64), np.ones(500, dtype=np.int64)])
X_imb = rng.normal(size=(len(y_imb), 6)).astype(np.float32)

pos_idx = np.flatnonzero(y_imb == 1)                       # positions where True
neg_idx = np.flatnonzero(y_imb == 0)
neg_sample = rng.choice(neg_idx, size=len(pos_idx), replace=False)

balanced_idx = np.concatenate([pos_idx, neg_sample])
rng.shuffle(balanced_idx)
X_bal, y_bal = X_imb[balanced_idx], y_imb[balanced_idx]
print("original class counts:", np.bincount(y_imb))
print("balanced class counts:", np.bincount(y_bal))

# Alternative that keeps all data: class weights inversely proportional to frequency.
counts = np.bincount(y_imb)
class_weights = counts.sum() / (len(counts) * counts)
print("class weights        :", np.round(class_weights, 3))

In [ ]:
# Hard example mining: train more on the samples the model gets wrong.
per_sample_loss = rng.random(1000).astype(np.float32)
k = 32
hardest = np.argpartition(-per_sample_loss, k)[:k]   # top k without a full sort
print("hardest sample losses:", np.round(np.sort(per_sample_loss[hardest])[-5:], 3))
print("argpartition is O(n), argsort is O(n log n). At a million samples that matters.")

---
# 7. Axes and reductions

## What it is

A reduction collapses one or more axes into a single value per remaining position. `axis=k` means **"the axis k disappears"**. That single sentence resolves almost every axis confusion.

For `X` of shape `(1000, 12)`:
- `X.sum(axis=0)` removes axis 0, leaving shape `(12,)`, one value per feature.
- `X.sum(axis=1)` removes axis 1, leaving shape `(1000,)`, one value per sample.
- `X.sum()` removes everything, leaving a 0D scalar.

`keepdims=True` keeps the collapsed axis with length 1, which is what makes the result broadcast cleanly back against the original.

## Why production cares

Every metric, every normalization, and every pooling operation is a reduction along a specific axis. Choosing the wrong axis rarely raises an error, it just produces a wrong number that flows into your dashboards and your model.

## Where it shows up

- **Feature scaling**: per feature mean and standard deviation, `axis=0` over samples.
- **Per sample aggregation**: mean pooling token embeddings into one sentence embedding, `axis=1` over the time axis.
- **Global average pooling** in a CNN: `axis=(2, 3)` over height and width, turning `(N, C, H, W)` into `(N, C)`.
- **Batch metrics**: accuracy is `(preds == labels).mean()`.
- **Predicted class**: `logits.argmax(axis=-1)`.
- **Per class metrics**: precision and recall from a confusion matrix are row and column sums.
- **Running totals and cumulative curves**: `cumsum` for a cumulative gains chart or a learning curve.
- **NaN safe statistics**: `nanmean`, `nansum` when sensors drop out.

In [ ]:
X = np.array([[1., 2., 3.],
              [4., 5., 6.],
              [7., 8., 9.]], dtype=np.float32)

print("X:\n", X)
print("\nX.sum()            :", X.sum(), " shape", np.shape(X.sum()), " all axes gone")
print("X.sum(axis=0)      :", X.sum(axis=0), " shape", X.sum(axis=0).shape, " axis 0 gone, per COLUMN")
print("X.sum(axis=1)      :", X.sum(axis=1), " shape", X.sum(axis=1).shape, " axis 1 gone, per ROW")
print("X.sum(axis=0, keepdims=True) shape:", X.sum(axis=0, keepdims=True).shape)
print("X.sum(axis=1, keepdims=True) shape:", X.sum(axis=1, keepdims=True).shape)

In [ ]:
# Why keepdims matters: row normalization in one expression.
row_sums = X.sum(axis=1, keepdims=True)      # (3, 1)
normalized = X / row_sums                    # broadcasts cleanly
print("row normalized:\n", np.round(normalized, 3))
print("rows sum to 1 :", np.allclose(normalized.sum(axis=1), 1.0))

# Without keepdims this raises or, worse, broadcasts the wrong way.
try:
    bad = X / X.sum(axis=1)                  # (3,3) / (3,) divides COLUMN wise
    print("\nwithout keepdims (silently wrong, no error):\n", np.round(bad, 3))
except ValueError as e:
    print("error:", e)

In [ ]:
# The standard scaler: fit on train only, apply everywhere.
X_train = rng.normal(loc=5.0, scale=2.0, size=(800, 4)).astype(np.float32)
X_test  = rng.normal(loc=5.0, scale=2.0, size=(200, 4)).astype(np.float32)

mu    = X_train.mean(axis=0)                 # (4,) per feature
sigma = X_train.std(axis=0) + 1e-8           # epsilon guards constant features

X_train_s = (X_train - mu) / sigma
X_test_s  = (X_test  - mu) / sigma           # NEVER refit on test, that is leakage

print("train mean after scaling:", np.round(X_train_s.mean(axis=0), 6))
print("train std  after scaling:", np.round(X_train_s.std(axis=0), 6))
print("test  mean after scaling:", np.round(X_test_s.mean(axis=0), 4), " (not exactly 0, correct)")
print("\nmu and sigma are model artifacts. Save them with the model or serving breaks.")

In [ ]:
# Reductions over multiple axes: global average pooling in a CNN head.
feature_maps = rng.random((32, 64, 7, 7)).astype(np.float32)   # (N, C, H, W)
pooled = feature_maps.mean(axis=(2, 3))                        # collapse H and W
print("feature maps:", feature_maps.shape)
print("pooled      :", pooled.shape, " one number per channel per sample")

# Mean pooling token embeddings into a sentence embedding, respecting padding.
tokens = rng.normal(size=(4, 10, 8)).astype(np.float32)        # (batch, seq, dim)
pad_mask = np.array([[1,1,1,1,1,0,0,0,0,0],
                     [1,1,1,1,1,1,1,1,0,0],
                     [1,1,0,0,0,0,0,0,0,0],
                     [1]*10], dtype=np.float32)                # 1 = real token

masked_sum = (tokens * pad_mask[..., None]).sum(axis=1)        # (4, 8)
lengths    = pad_mask.sum(axis=1, keepdims=True)               # (4, 1)
sentence_emb = masked_sum / lengths
print("\nsentence embeddings:", sentence_emb.shape)
print("naive tokens.mean(axis=1) would average in the padding and corrupt short sequences.")

In [ ]:
# argmax, argmin, and the metrics built from them.
logits = rng.normal(size=(6, 4)).astype(np.float32)
preds  = logits.argmax(axis=-1)                 # axis=-1 works for any rank
labels = rng.integers(0, 4, size=6)

print("logits shape:", logits.shape)
print("predictions :", preds)
print("labels      :", labels)
print("accuracy    :", (preds == labels).mean())

# Top-1 confidence per sample.
probs = np.exp(logits - logits.max(axis=-1, keepdims=True))
probs /= probs.sum(axis=-1, keepdims=True)
print("confidence  :", np.round(probs.max(axis=-1), 3))

# Which epoch had the best validation loss.
val_losses = np.array([0.9, 0.7, 0.55, 0.58, 0.61])
print("\nbest epoch  :", val_losses.argmin(), " best loss:", val_losses.min())

In [ ]:
# Per class precision and recall straight from a confusion matrix.
cm = np.array([[50,  2,  3],
               [ 4, 40,  6],
               [ 1,  5, 39]], dtype=np.float64)     # rows = true, cols = predicted

tp = np.diag(cm)
precision = tp / cm.sum(axis=0)     # column sums are total predicted per class
recall    = tp / cm.sum(axis=1)     # row sums are total actual per class
f1 = 2 * precision * recall / (precision + recall)

print("precision:", np.round(precision, 3))
print("recall   :", np.round(recall, 3))
print("f1       :", np.round(f1, 3))
print("macro f1 :", round(f1.mean(), 3))
print("accuracy :", round(tp.sum() / cm.sum(), 3))

In [ ]:
# NaN safe reductions: sensors fail, columns go missing, joins produce nulls.
sensor = np.array([[1.0, 2.0, np.nan],
                   [4.0, np.nan, 6.0],
                   [7.0, 8.0, 9.0]], dtype=np.float32)

print("mean     :", sensor.mean(axis=0), " one NaN poisons the whole column")
print("nanmean  :", np.nanmean(sensor, axis=0))
print("nanstd   :", np.round(np.nanstd(sensor, axis=0), 3))
print("nan count per column:", np.isnan(sensor).sum(axis=0))

# cumsum for cumulative curves.
daily_revenue = np.array([100, 150, 90, 200, 175], dtype=np.float64)
print("\ncumulative revenue:", np.cumsum(daily_revenue))

---
# 8. Broadcasting

## What it is

Broadcasting lets NumPy operate on arrays of different shapes by virtually stretching the smaller one, without ever materialising the copy.

**The rules.** Align shapes from the right. For each axis pair, they are compatible if they are equal, or if one of them is 1. A missing leading axis is treated as 1. If any pair fails, you get a `ValueError`.

```
(3, 4)  +  (4,)      ->  (4,) becomes (1,4) becomes (3,4)   OK
(3, 4)  +  (3, 1)    ->  (3,1) stretches along axis 1        OK
(3, 4)  +  (3,)      ->  aligns as (3,4) vs (1,3), 4 vs 3    ValueError
(8,1,6,1) + (7,1,5)  ->  (8,7,6,5)                           OK
```

## Why production cares

Broadcasting is what makes vectorized normalization, bias addition, and masking possible without loops or wasted memory. It is also the source of the most dangerous class of silent bug in machine learning code, because two wrong shapes can be **compatible**, so no error is raised and your loss quietly becomes a matrix.

## Where it shows up

- **Standardization**: `(X - mu) / sigma` where `X` is `(n, d)` and `mu` is `(d,)`.
- **Bias addition** in a dense layer: `(batch, out) + (out,)`.
- **Per channel image normalization** with ImageNet statistics: `(N,3,H,W) - (3,1,1)`.
- **Attention masking**: adding a `(1, 1, seq, seq)` mask to `(batch, heads, seq, seq)` scores.
- **Pairwise distance matrices** for k nearest neighbours, clustering, and contrastive losses: `(n,1,d) - (1,m,d)` gives `(n,m,d)`.
- **Sample weights** applied to a per sample loss vector.
- **Positional encodings** added to token embeddings.

In [61]:
a=np.array([[1,2,3],
           [5,6,7]])

b=np.array([1,2,3])

In [65]:
a*b

array([[ 1,  4,  9],
       [ 5, 12, 21]])

In [64]:
a.shape,b.shape

((2, 3), (3,))

In [55]:
one_d=np.array([100,101,102,103])

In [56]:
one_d

array([100, 101, 102, 103])

In [60]:
one_d+np.array([10,10,10,10])

array([110, 111, 112, 113])

In [59]:
one_d+10

array([110, 111, 112, 113])

In [ ]:
# Rule walkthrough.
def try_broadcast(s1, s2):
    try:
        out = np.broadcast_shapes(s1, s2)
        print(f"{str(s1):14s} with {str(s2):14s} -> {out}")
    except ValueError as e:
        print(f"{str(s1):14s} with {str(s2):14s} -> ValueError: {e}")

try_broadcast((3, 4), (4,))
try_broadcast((3, 4), (3, 1))
try_broadcast((3, 4), (3,))
try_broadcast((8, 1, 6, 1), (7, 1, 5))
try_broadcast((32, 3, 224, 224), (3, 1, 1))
try_broadcast((32, 12, 512, 512), (1, 1, 512, 512))

In [ ]:
# Broadcasting costs no memory for the stretched operand.
X = rng.normal(size=(100_000, 20)).astype(np.float32)
mu = X.mean(axis=0)                    # (20,), 80 bytes
Xs = X - mu                            # mu is NOT expanded to (100000, 20)

print("X   :", X.nbytes / 1e6, "MB")
print("mu  :", mu.nbytes, "bytes")
print("A loop or an explicit np.tile(mu, (100000,1)) would waste",
      f"{X.nbytes/1e6:.1f} MB.")

In [ ]:
# Per channel image normalization, the standard vision preprocessing step.
images = rng.random((32, 3, 224, 224)).astype(np.float32)   # NCHW in [0,1]

imagenet_mean = np.array([0.485, 0.456, 0.406], dtype=np.float32).reshape(3, 1, 1)
imagenet_std  = np.array([0.229, 0.224, 0.225], dtype=np.float32).reshape(3, 1, 1)

normalized = (images - imagenet_mean) / imagenet_std
print("images    :", images.shape)
print("mean      :", imagenet_mean.shape, "broadcasts over N, H, W")
print("normalized:", normalized.shape)
print("per channel mean after:", np.round(normalized.mean(axis=(0, 2, 3)), 3))

# For NHWC the reshape is different. Getting this wrong normalizes the wrong axis.
images_nhwc = np.transpose(images, (0, 2, 3, 1))
mean_nhwc = imagenet_mean.reshape(1, 1, 1, 3)
print("\nNHWC needs mean shaped:", mean_nhwc.shape, "not", imagenet_mean.shape)

In [ ]:
# Pairwise Euclidean distances with broadcasting: the core of k-NN and clustering.
queries = rng.normal(size=(5, 8)).astype(np.float32)
database = rng.normal(size=(100, 8)).astype(np.float32)

diff = queries[:, None, :] - database[None, :, :]     # (5, 1, 8) - (1, 100, 8) -> (5, 100, 8)
dists = np.sqrt((diff ** 2).sum(axis=-1))             # (5, 100)
print("distance matrix:", dists.shape)
print("nearest neighbour per query:", dists.argmin(axis=1))

# MEMORY WARNING: the intermediate `diff` is n*m*d elements.
n, m, d = 10_000, 100_000, 128
print(f"\nFor n={n}, m={m}, d={d} that intermediate is "
      f"{n*m*d*4/1e12:.1f} TB. It will not fit.")

# The production form uses the identity ||a-b||^2 = ||a||^2 + ||b||^2 - 2ab,
# which never materialises the 3D intermediate.
q2 = (queries ** 2).sum(axis=1, keepdims=True)        # (5, 1)
d2 = (database ** 2).sum(axis=1)                      # (100,)
dists_fast = np.sqrt(np.maximum(q2 + d2 - 2 * queries @ database.T, 0))
print("matches naive version:", np.allclose(dists, dists_fast, atol=1e-4))
print("intermediate is only (n, m), and the work is one matmul (BLAS optimized).")

In [ ]:
# THE SILENT BUG. This is the one that ships to production.
y_true = np.array([1.0, 2.0, 3.0, 4.0])            # shape (4,)
y_pred = np.array([[1.1], [2.1], [2.9], [4.2]])    # shape (4, 1), a common model output

errors = y_true - y_pred
print("y_true shape :", y_true.shape)
print("y_pred shape :", y_pred.shape)
print("errors shape :", errors.shape, "  <- expected (4,), got a 4x4 MATRIX")
print(errors)
print("\nmse computed  :", (errors ** 2).mean(), " <- wrong, averaged over 16 numbers")
print("mse correct   :", ((y_true - y_pred.ravel()) ** 2).mean())
print("\nNo exception was raised. Your loss curve just looks slightly off, forever.")

In [ ]:
# Defence: assert shapes at function boundaries in numerical code.
def mse(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    assert y_true.shape == y_pred.shape, f"shape mismatch: {y_true.shape} vs {y_pred.shape}"
    return float(((y_true - y_pred) ** 2).mean())

print("correct call:", mse(y_true, y_pred.ravel()))
try:
    mse(y_true, y_pred)
except AssertionError as e:
    print("caught early :", e)

In [ ]:
# Sample weights and attention masks, two more everyday broadcasts.
per_sample_loss = np.array([0.5, 1.2, 0.3, 2.0], dtype=np.float32)   # (4,)
weights = np.array([1.0, 3.0, 1.0, 0.5], dtype=np.float32)           # (4,)
weighted = (per_sample_loss * weights).sum() / weights.sum()
print("weighted loss:", round(float(weighted), 4))

# Attention mask broadcast over batch and heads.
batch, heads, seq = 4, 12, 6
scores = rng.normal(size=(batch, heads, seq, seq)).astype(np.float32)
causal = np.triu(np.ones((seq, seq), dtype=bool), k=1)               # (seq, seq)
scores_masked = np.where(causal[None, None, :, :], -1e9, scores)     # broadcasts to (4,12,6,6)
print("scores       :", scores.shape)
print("mask         :", causal.shape, "broadcast to", scores_masked.shape)
print("row 0 of head 0 after masking:\n", np.round(scores_masked[0, 0, 0], 2))

---
# 9. Vectorization

## What it is

Replacing an explicit Python loop with an array expression that runs in compiled code. NumPy provides **ufuncs** (universal functions) that apply elementwise: `np.exp`, `np.log`, `np.sqrt`, `np.maximum`, and every arithmetic operator.

The mental shift: stop thinking "for each element, do X" and start thinking "apply X to the whole array".

## Why production cares

A vectorized rewrite is routinely 20x to 200x faster, with no algorithmic change. In a data loader that difference decides whether your GPU is saturated or idle. In a nightly batch job it decides whether you finish before the business day starts.

`np.vectorize` is **not** vectorization. It is a convenience wrapper that still loops in Python. It does not make anything fast.

## Where it shows up

- **Activation functions**: ReLU is `np.maximum(x, 0)`, sigmoid is `1/(1+np.exp(-x))`.
- **Feature engineering**: ratios, logs, differences, interaction terms over an entire column at once.
- **Gradient clipping**: `np.clip(grads, -1, 1)`.
- **Conditional logic**: `np.where` and `np.select` instead of if statements inside a loop.
- **Business rules**: tiered pricing, risk banding, bucketing ages into cohorts.
- **Image augmentation**: brightness, contrast, and normalization applied to a whole batch.

In [ ]:
# Loop versus vectorized, on a realistic feature computation.
n = 500_000
values = rng.random(n).astype(np.float32) * 100

def loop_version(v):
    out = np.empty_like(v)
    for i in range(len(v)):
        x = v[i]
        out[i] = np.log1p(x) * 2.0 if x > 50 else np.sqrt(x)
    return out

def vectorized_version(v):
    return np.where(v > 50, np.log1p(v) * 2.0, np.sqrt(v))

t0 = time.perf_counter(); r1 = loop_version(values);       t_loop = time.perf_counter() - t0
t0 = time.perf_counter(); r2 = vectorized_version(values); t_vec  = time.perf_counter() - t0

print(f"loop       : {t_loop*1000:8.1f} ms")
print(f"vectorized : {t_vec*1000:8.1f} ms")
print(f"speedup    : {t_loop/t_vec:8.1f}x")
print("same result:", np.allclose(r1, r2))

In [ ]:
# np.vectorize is a trap. It is syntax sugar, not speed.
def scalar_fn(x):
    return x * 2 + 1

vfn = np.vectorize(scalar_fn)
small = rng.random(200_000)

t0 = time.perf_counter(); vfn(small);         t_vfn = time.perf_counter() - t0
t0 = time.perf_counter(); small * 2 + 1;      t_real = time.perf_counter() - t0
print(f"np.vectorize     : {t_vfn*1000:7.1f} ms")
print(f"real vectorization: {t_real*1000:7.1f} ms")
print(f"np.vectorize is {t_vfn/t_real:.0f}x slower. Use it only for convenience, never for speed.")

In [ ]:
# The activation function toolkit, all vectorized.
x = np.linspace(-5, 5, 9, dtype=np.float32)

relu       = np.maximum(x, 0)
leaky_relu = np.where(x > 0, x, 0.01 * x)
sigmoid    = 1.0 / (1.0 + np.exp(-x))
tanh       = np.tanh(x)
# GELU, the activation used in transformers (tanh approximation).
gelu = 0.5 * x * (1 + np.tanh(np.sqrt(2/np.pi) * (x + 0.044715 * x**3)))

print("x          :", np.round(x, 2))
print("relu       :", np.round(relu, 2))
print("leaky relu :", np.round(leaky_relu, 3))
print("sigmoid    :", np.round(sigmoid, 3))
print("tanh       :", np.round(tanh, 3))
print("gelu       :", np.round(gelu, 3))

In [ ]:
# np.select for multi branch business logic, no loops.
ages = np.array([15, 22, 35, 47, 63, 78], dtype=np.int32)
conditions = [ages < 18, ages < 30, ages < 50, ages < 65]
choices    = ["minor", "young_adult", "adult", "middle_age"]
cohort = np.select(conditions, choices, default="senior")
print("ages   :", ages)
print("cohorts:", cohort)

# Tiered pricing, a common feature engineering task.
usage = np.array([50, 500, 5_000, 50_000], dtype=np.float64)
rate = np.select([usage < 100, usage < 1000, usage < 10_000],
                 [0.10, 0.08, 0.05], default=0.03)
print("\nusage:", usage, " rate:", rate, " cost:", usage * rate)

In [ ]:
# Gradient clipping, both flavours, as used in every RNN and transformer trainer.
grads = rng.normal(size=(1000,)).astype(np.float32) * 10

# Value clipping: bound every element.
clipped_val = np.clip(grads, -1.0, 1.0)

# Norm clipping: rescale the whole vector if its L2 norm is too large.
max_norm = 5.0
norm = np.linalg.norm(grads)
scale = min(1.0, max_norm / (norm + 1e-6))
clipped_norm = grads * scale

print("original norm :", round(float(norm), 2))
print("after value clip norm:", round(float(np.linalg.norm(clipped_val)), 2))
print("after norm  clip norm:", round(float(np.linalg.norm(clipped_norm)), 2))
print("\nNorm clipping preserves gradient DIRECTION, value clipping does not.")

In [ ]:
# In place operations and out= avoid allocating new arrays in hot loops.
big = rng.random((2000, 2000)).astype(np.float32)

t0 = time.perf_counter()
for _ in range(5):
    big2 = big * 2.0 + 1.0          # allocates a new 16 MB array each iteration
t_alloc = time.perf_counter() - t0

buf = np.empty_like(big)
t0 = time.perf_counter()
for _ in range(5):
    np.multiply(big, 2.0, out=buf)  # reuses one buffer
    np.add(buf, 1.0, out=buf)
t_inplace = time.perf_counter() - t0

print(f"allocating : {t_alloc*1000:7.1f} ms")
print(f"out= reuse : {t_inplace*1000:7.1f} ms")
print("In a data loader running thousands of times per epoch, this is real throughput.")

---
# 10. Linear algebra

## What it is

Matrix multiplication and its relatives. In NumPy the operator is `@` (equivalently `np.matmul`). For 2D arrays it is the familiar matrix product. For higher dimensional arrays it treats the last two axes as the matrix and **batches over all the leading axes**, which is exactly what deep learning needs.

`np.dot` behaves differently for ndim greater than 2 and should be avoided in new code. Use `@` for matrices and `einsum` when the index bookkeeping gets complicated.

## Why production cares

Matrix multiply is where essentially all of a neural network's compute goes. It is also the operation with the most optimized implementations on the planet, through BLAS libraries such as OpenBLAS and Intel MKL. Writing your computation as a matmul means you get multithreaded, cache blocked, SIMD accelerated code for free. Writing it as a loop means you get none of that.

## Where it shows up

- **A dense layer forward pass**: `X @ W + b`.
- **Attention**: `softmax(Q @ K.T / sqrt(d)) @ V`, batched over samples and heads.
- **Cosine similarity search**: normalize the embeddings, then one matmul gives every pairwise similarity.
- **Linear regression** by normal equations or `lstsq`.
- **PCA** by SVD, for dimensionality reduction and visualisation.
- **Covariance matrices** for anomaly detection with Mahalanobis distance.
- **Graph algorithms**: adjacency matrix powers count paths, and message passing in a GNN is a sparse matmul.

In [ ]:
# The operators, and when each applies.
A = rng.normal(size=(3, 4)).astype(np.float32)
B = rng.normal(size=(4, 2)).astype(np.float32)

print("A @ B           :", (A @ B).shape,          " matrix product")
print("np.matmul(A, B) :", np.matmul(A, B).shape,  " identical")
print("A * A           :", (A * A).shape,          " ELEMENTWISE, a different operation")

v = rng.normal(size=4).astype(np.float32)
print("A @ v           :", (A @ v).shape,          " matrix times vector")
print("v @ v           :", float(v @ v),           " inner product, a scalar")
print("np.outer(v, v)  :", np.outer(v, v).shape,   " outer product")

In [ ]:
# A two layer neural network forward pass, entirely in NumPy.
batch, d_in, d_hidden, n_classes = 16, 20, 64, 3

X  = rng.normal(size=(batch, d_in)).astype(np.float32)
W1 = (rng.normal(size=(d_in, d_hidden)) * np.sqrt(2.0 / d_in)).astype(np.float32)   # He init
b1 = np.zeros(d_hidden, dtype=np.float32)
W2 = (rng.normal(size=(d_hidden, n_classes)) * np.sqrt(2.0 / d_hidden)).astype(np.float32)
b2 = np.zeros(n_classes, dtype=np.float32)

h = np.maximum(X @ W1 + b1, 0)      # linear then ReLU
logits = h @ W2 + b2

print("X      :", X.shape)
print("hidden :", h.shape)
print("logits :", logits.shape)
print("params :", W1.size + b1.size + W2.size + b2.size)

In [ ]:
# Batched matmul: matmul treats the last two axes as the matrix.
Q = rng.normal(size=(8, 12, 64, 32)).astype(np.float32)   # (batch, heads, seq, head_dim)
K = rng.normal(size=(8, 12, 64, 32)).astype(np.float32)
V = rng.normal(size=(8, 12, 64, 32)).astype(np.float32)

d_k = Q.shape[-1]
scores = (Q @ K.transpose(0, 1, 3, 2)) / np.sqrt(d_k)      # (8, 12, 64, 64)

# Stable softmax over the last axis (Section 11 explains the max subtraction).
scores -= scores.max(axis=-1, keepdims=True)
weights = np.exp(scores)
weights /= weights.sum(axis=-1, keepdims=True)

out = weights @ V                                          # (8, 12, 64, 32)
print("Q, K, V :", Q.shape)
print("scores  :", scores.shape, " every token against every token, per head")
print("output  :", out.shape)
print("attention rows sum to 1:", np.allclose(weights.sum(axis=-1), 1.0, atol=1e-5))

In [ ]:
# Cosine similarity search, the operation behind every vector database.
docs    = rng.normal(size=(5000, 384)).astype(np.float32)
queries = rng.normal(size=(3, 384)).astype(np.float32)

# Normalize once, then cosine similarity is just a dot product.
docs_n    = docs    / (np.linalg.norm(docs,    axis=1, keepdims=True) + 1e-12)
queries_n = queries / (np.linalg.norm(queries, axis=1, keepdims=True) + 1e-12)

sims = queries_n @ docs_n.T                # (3, 5000)
k = 5
top_k = np.argpartition(-sims, k, axis=1)[:, :k]
top_k = np.take_along_axis(top_k, np.argsort(-np.take_along_axis(sims, top_k, 1), axis=1), axis=1)

print("similarity matrix:", sims.shape)
print("top 5 doc ids per query:\n", top_k)
print("their scores:\n", np.round(np.take_along_axis(sims, top_k, axis=1), 3))
print("\nNormalizing the index once at build time turns every search into one matmul.")

In [ ]:
# einsum: name your axes and let NumPy figure out the loops.
# This is the readable way to express contractions you would otherwise
# implement with a stack of transpose and reshape calls.
Xe = rng.normal(size=(32, 10, 64)).astype(np.float32)     # (batch, seq, dim)
We = rng.normal(size=(64, 128)).astype(np.float32)        # (dim, out)

out_einsum = np.einsum("bsd,do->bso", Xe, We)
out_matmul = Xe @ We
print("einsum result matches matmul:", np.allclose(out_einsum, out_matmul, atol=1e-4))

print("\ncommon einsum patterns:")
M = rng.normal(size=(4, 4)).astype(np.float32)
print("  trace          np.einsum('ii->', M)      :", round(float(np.einsum("ii->", M)), 4))
print("  diagonal       np.einsum('ii->i', M)     :", np.round(np.einsum("ii->i", M), 3))
print("  transpose      np.einsum('ij->ji', M)    :", np.einsum("ij->ji", M).shape)
print("  row sums       np.einsum('ij->i', M)     :", np.round(np.einsum("ij->i", M), 3))
print("  batched matmul np.einsum('bij,bjk->bik') : contracts j, keeps batch b")
print("  attention      np.einsum('bhqd,bhkd->bhqk', Q, K) : scores without a transpose call")

In [ ]:
# Solving linear systems. Never invert a matrix if you can solve instead.
Xr = rng.normal(size=(200, 5)).astype(np.float64)
true_w = np.array([1.5, -2.0, 0.5, 3.0, -1.0])
yr = Xr @ true_w + rng.normal(size=200) * 0.1

# Normal equations with solve: faster and numerically better than using inv.
w_solve = np.linalg.solve(Xr.T @ Xr, Xr.T @ yr)

# lstsq handles rank deficient and non square cases safely.
w_lstsq, residuals, rank, sv = np.linalg.lstsq(Xr, yr, rcond=None)

print("true       :", np.round(true_w, 3))
print("solve      :", np.round(w_solve, 3))
print("lstsq      :", np.round(w_lstsq, 3))
print("matrix rank:", rank, " condition number:", round(float(np.linalg.cond(Xr.T @ Xr)), 1))
print("\nA large condition number (say above 1e10) means your features are nearly")
print("collinear and the solution is unstable. Add regularization or drop features.")

In [ ]:
# PCA via SVD: dimensionality reduction, denoising, and visualisation.
# Realistic case: 20 observed features driven by only 4 latent factors plus noise.
latent   = rng.normal(size=(500, 4))
loadings = rng.normal(size=(4, 20))
Xp = (latent @ loadings + 0.3 * rng.normal(size=(500, 20))).astype(np.float32)

Xc = Xp - Xp.mean(axis=0)                      # centring is mandatory for PCA
U, S, Vt = np.linalg.svd(Xc, full_matrices=False)

explained = S ** 2 / (S ** 2).sum()
print("explained variance ratio (first 6):", np.round(explained[:6], 4))
print("cumulative                        :", np.round(np.cumsum(explained[:6]), 4))

n_components = 3
X_reduced = Xc @ Vt[:n_components].T           # project onto the top components
print("\noriginal :", Xp.shape)
print("reduced  :", X_reduced.shape)

X_recon = X_reduced @ Vt[:n_components]        # reconstruct, lossy
print("reconstruction error:", round(float(np.linalg.norm(Xc - X_recon) / np.linalg.norm(Xc)), 4))
print("\nProduction uses: compressing embeddings before storing them in a vector DB,")
print("2D plots of a high dimensional space, and denoising by dropping small components.")

In [ ]:
# Norms, and the regularization penalties built from them.
w = rng.normal(size=(64, 32)).astype(np.float32)

print("L2 (Frobenius) norm:", round(float(np.linalg.norm(w)), 4))
print("L1 norm            :", round(float(np.abs(w).sum()), 4))
print("max abs weight     :", round(float(np.abs(w).max()), 4))
print("per row L2 norms   :", np.round(np.linalg.norm(w, axis=1)[:5], 3))

l2_penalty = 1e-4 * (w ** 2).sum()          # weight decay, shrinks weights smoothly
l1_penalty = 1e-4 * np.abs(w).sum()         # lasso, drives weights to exactly zero
print("\nL2 penalty:", round(float(l2_penalty), 5), " L1 penalty:", round(float(l1_penalty), 5))

---
# 11. Numerical stability

## What it is

Floating point numbers have finite range and finite precision. Operations that are fine on paper overflow, underflow, or lose all their significant digits on a computer. Numerical stability is the practice of rearranging a formula so it stays inside the representable range.

## Why production cares

"Loss became NaN at step 4000" is one of the most common and most expensive training failures. Once a NaN enters, it propagates through every subsequent operation and the run is dead. The causes are almost always one of: `exp` of a large number, `log` of zero, division by zero, or a square root of a negative number produced by cancellation.

## The three fixes you will use constantly

1. **Subtract the max before `exp`.** This is the log sum exp trick and it is why every real softmax implementation has a `- max` in it.
2. **Add an epsilon before dividing or taking a log.** Typically `1e-8` for float32, `1e-12` for float64.
3. **Clip inputs to a known safe range** before feeding them into `log` or `exp`.

## Where it shows up

- Softmax and cross entropy in every classifier.
- Batch, layer, and RMS normalization, all of which divide by a standard deviation.
- Variance computed as `E[x^2] - E[x]^2`, which suffers catastrophic cancellation.
- Log probabilities in language models, beam search, and hidden Markov models.
- Metric computations that divide by a class count that can be zero.

In [ ]:
# Naive softmax overflows. This is not hypothetical, logits reach these values.
def softmax_naive(x):
    e = np.exp(x)
    return e / e.sum()

def softmax_stable(x, axis=-1):
    x = x - np.max(x, axis=axis, keepdims=True)     # the one line that matters
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)

logits_big = np.array([1000.0, 1001.0, 1002.0], dtype=np.float32)
with np.errstate(over="ignore", invalid="ignore"):
    print("naive :", softmax_naive(logits_big), " <- overflow to inf then nan")
print("stable:", softmax_stable(logits_big))

logits_small = np.array([-1000.0, -1001.0, -1002.0], dtype=np.float32)
with np.errstate(under="ignore", invalid="ignore"):
    print("\nnaive on very negative :", softmax_naive(logits_small), " <- underflow to 0/0")
print("stable on very negative:", softmax_stable(logits_small))

In [ ]:
# Cross entropy done safely: work in log space, never take log of a probability
# you computed with exp.
def log_softmax(x, axis=-1):
    x_max = np.max(x, axis=axis, keepdims=True)
    shifted = x - x_max
    return shifted - np.log(np.exp(shifted).sum(axis=axis, keepdims=True))

def cross_entropy(logits, labels):
    logp = log_softmax(logits, axis=-1)
    return -logp[np.arange(len(labels)), labels].mean()

logits = np.array([[2.0, 1.0, 0.1],
                   [0.5, 3.0, 0.2],
                   [1.0, 1.0, 5.0]], dtype=np.float32)
labels = np.array([0, 1, 2])
print("cross entropy:", round(float(cross_entropy(logits, labels)), 5))

# The unsafe version, for comparison: probability 0 gives log(0) = -inf.
probs = softmax_stable(np.array([[0.0, 0.0, 100.0]], dtype=np.float32))
with np.errstate(divide="ignore"):
    print("log of an underflowed probability:", np.log(probs))
print("clipped instead                   :", np.log(np.clip(probs, 1e-12, 1.0)))

In [ ]:
# Division guards. Every normalization layer needs one.
x = np.array([5.0, 5.0, 5.0, 5.0], dtype=np.float32)   # a constant feature
std = x.std()
print("std of a constant feature:", std)
with np.errstate(invalid="ignore"):
    print("without epsilon:", (x - x.mean()) / std, " <- nan")
print("with epsilon   :", (x - x.mean()) / (std + 1e-8))

# Recall and precision when a class never appears.
tp, fn, fp = 0, 0, 0
eps = 1e-12
print("\nrecall with guard   :", tp / (tp + fn + eps))
print("precision with guard:", tp / (tp + fp + eps))

In [ ]:
# Catastrophic cancellation: the textbook variance formula is unsafe.
data = np.array([1e8 + 1, 1e8 + 2, 1e8 + 3], dtype=np.float32)

var_naive = (data ** 2).mean() - data.mean() ** 2     # E[x^2] - E[x]^2
var_stable = ((data - data.mean()) ** 2).mean()       # two pass, what np.var does

print("naive variance :", var_naive, " <- can even go negative")
print("stable variance:", var_stable)
print("np.var         :", data.var())
print("\nSubtracting two nearly equal large numbers destroys the significant digits.")

In [ ]:
# NaN and Inf hunting: put these checks in your training loop.
def check_finite(name, arr):
    arr = np.asarray(arr)
    n_nan = int(np.isnan(arr).sum())
    n_inf = int(np.isinf(arr).sum())
    status = "OK" if (n_nan == 0 and n_inf == 0) else "PROBLEM"
    print(f"{name:14s} {status:8s} nan={n_nan:4d} inf={n_inf:4d} "
          f"min={np.nanmin(arr):+.3e} max={np.nanmax(arr):+.3e}")

good = rng.normal(size=1000).astype(np.float32)
bad  = good.copy()
bad[10] = np.nan
bad[20] = np.inf

check_finite("gradients", good)
check_finite("corrupted", bad)

# Turn a silent NaN into a loud exception while debugging.
with np.errstate(divide="raise", invalid="raise"):
    try:
        _ = np.float32(1.0) / np.float32(0.0)
    except FloatingPointError as e:
        print("\ncaught early:", e)
print("np.errstate(all='raise') around a training step tells you the EXACT line.")

---
# 12. Randomness and reproducibility

## What it is

NumPy has two random APIs. The legacy one (`np.random.seed`, `np.random.rand`) uses a single hidden global state. The modern one (`np.random.default_rng()`) gives you an explicit `Generator` object that you pass around.

Use the modern API. It is faster, statistically better, and it does not have a global that any imported library can silently reseed.

## Why production cares

Reproducibility is not academic tidiness. It is how you:

- prove that a model change caused a metric change, rather than seed noise,
- reproduce a bug reported by a teammate,
- pass an audit or regulatory review for a credit or medical model,
- get identical train and test splits every run so your evaluation is honest.

Global seeding also breaks in parallel data loading. If every worker process inherits the same global state, every worker generates **the same augmentations**, which silently reduces your effective data diversity.

## Where it shows up

- Train, validation, and test splitting.
- Weight initialisation (He for ReLU networks, Xavier for tanh).
- Dropout masks.
- Data augmentation: random crops, flips, noise, mixup coefficients.
- Bootstrap confidence intervals for a metric.
- Negative sampling in recommendation and contrastive training.
- Synthetic data generation for tests.

In [ ]:
# Legacy versus modern.
np.random.seed(0)                      # global, affects everything, avoid in libraries
legacy = np.random.rand(3)

gen_a = np.random.default_rng(0)       # explicit, local, safe
gen_b = np.random.default_rng(0)
print("legacy       :", np.round(legacy, 4))
print("generator a  :", np.round(gen_a.random(3), 4))
print("generator b  :", np.round(gen_b.random(3), 4), " identical to a, same seed")

# Independent streams for independent concerns, from one master seed.
master = np.random.SeedSequence(42)
split_rng, aug_rng, init_rng = [np.random.default_rng(s) for s in master.spawn(3)]
print("\nsplit stream :", np.round(split_rng.random(2), 4))
print("aug stream   :", np.round(aug_rng.random(2), 4))
print("init stream  :", np.round(init_rng.random(2), 4))
print("Changing your augmentation code no longer changes your data split.")

In [ ]:
# Reproducible splitting via a permutation.
def train_val_test_split(n, seed=42, fractions=(0.7, 0.15, 0.15)):
    g = np.random.default_rng(seed)
    perm = g.permutation(n)
    n_tr = int(fractions[0] * n)
    n_va = int((fractions[0] + fractions[1]) * n)
    return perm[:n_tr], perm[n_tr:n_va], perm[n_va:]

tr, va, te = train_val_test_split(1000, seed=42)
tr2, va2, te2 = train_val_test_split(1000, seed=42)
print("sizes            :", len(tr), len(va), len(te))
print("reproducible     :", np.array_equal(tr, tr2))
print("no leakage overlap:", len(np.intersect1d(tr, te)) == 0)

# Time series data must be split by TIME, never shuffled, or you leak the future.
timestamps = np.arange(1000)
cut = int(0.8 * len(timestamps))
print("\ntime split: train up to t =", cut, ", test from t =", cut)

In [ ]:
# Weight initialisation. The scale is not arbitrary, it keeps activation
# variance stable across layers so gradients neither vanish nor explode.
fan_in, fan_out = 256, 128
g = np.random.default_rng(0)

he      = g.normal(0, np.sqrt(2.0 / fan_in), size=(fan_in, fan_out)).astype(np.float32)
xavier  = g.normal(0, np.sqrt(2.0 / (fan_in + fan_out)), size=(fan_in, fan_out)).astype(np.float32)
uniform = g.uniform(-np.sqrt(6/(fan_in+fan_out)), np.sqrt(6/(fan_in+fan_out)),
                    size=(fan_in, fan_out)).astype(np.float32)

print(f"He (ReLU nets)   std: {he.std():.4f}  target {np.sqrt(2/fan_in):.4f}")
print(f"Xavier (tanh)    std: {xavier.std():.4f}  target {np.sqrt(2/(fan_in+fan_out)):.4f}")
print(f"Xavier uniform   std: {uniform.std():.4f}")
print("\nInitialising everything to zero makes all units identical forever.")
print("Initialising too large saturates activations and kills the gradient.")

In [ ]:
# Dropout, implemented as inverted dropout (scale at train time, no op at inference).
def dropout(x, p, generator, training=True):
    if not training or p == 0.0:
        return x
    keep = 1.0 - p
    mask = (generator.random(x.shape) < keep).astype(x.dtype) / keep
    return x * mask

acts = np.ones((4, 8), dtype=np.float32)
g = np.random.default_rng(1)
train_out = dropout(acts, 0.5, g, training=True)
eval_out  = dropout(acts, 0.5, g, training=False)
print("training output (some zeroed, survivors scaled by 1/0.5):\n", train_out)
print("\nexpected value preserved:", round(float(train_out.mean()), 3), "vs", float(eval_out.mean()))

In [ ]:
# Bootstrap confidence interval: how much of your metric is noise?
y_true = rng.integers(0, 2, size=500)
y_pred = y_true.copy()
flip = rng.choice(500, size=60, replace=False)
y_pred[flip] = 1 - y_pred[flip]              # 88 percent accuracy

point = (y_true == y_pred).mean()

g = np.random.default_rng(7)
n_boot = 2000
idx = g.integers(0, 500, size=(n_boot, 500))          # resample with replacement
boot_acc = (y_true[idx] == y_pred[idx]).mean(axis=1)  # fully vectorized, no loop
lo, hi = np.percentile(boot_acc, [2.5, 97.5])

print(f"accuracy      : {point:.4f}")
print(f"95% CI        : [{lo:.4f}, {hi:.4f}]")
print(f"CI width      : {hi-lo:.4f}")
print("\nIf a competing model is 0.5 points better and the CI is 3 points wide,")
print("you have not shown an improvement.")

In [ ]:
# Data augmentation on a batch, all vectorized.
g = np.random.default_rng(123)
images = g.random((8, 3, 32, 32)).astype(np.float32)

flip = g.random(8) < 0.5                                   # per sample coin flip
images_aug = np.where(flip[:, None, None, None], images[..., ::-1], images)

brightness = g.uniform(0.8, 1.2, size=(8, 1, 1, 1)).astype(np.float32)
images_aug = np.clip(images_aug * brightness, 0, 1)

noise = g.normal(0, 0.01, size=images_aug.shape).astype(np.float32)
images_aug = np.clip(images_aug + noise, 0, 1)

print("flipped samples:", np.flatnonzero(flip))
print("output shape   :", images_aug.shape, " range:",
      round(float(images_aug.min()), 3), "to", round(float(images_aug.max()), 3))

# mixup: blend pairs of samples and their labels.
lam = g.beta(0.2, 0.2, size=(8, 1, 1, 1)).astype(np.float32)
perm = g.permutation(8)
mixed = lam * images + (1 - lam) * images[perm]
print("mixup coefficients:", np.round(lam.ravel(), 3))

---
# 13. Reshaping and combining arrays

## What it is

Changing how the same buffer is interpreted (`reshape`, `transpose`, `expand_dims`, `squeeze`), and building bigger arrays out of smaller ones (`concatenate`, `stack`, `split`, `pad`, `repeat`, `tile`).

Two distinctions matter:

- `reshape` reinterprets the same data in a new shape and usually returns a **view**. `transpose` returns a view with different strides. Neither moves data unless it has to.
- `concatenate` joins along an **existing** axis, so ndim stays the same. `stack` creates a **new** axis, so ndim increases by one. This is the single most common mix up.

## Where it shows up

- **Flattening** a convolutional feature map before a dense head: `(N, C, H, W)` to `(N, C*H*W)`.
- **Collating a batch**: `np.stack([sample1, sample2, ...])` in a data loader.
- **Appending features**: `np.concatenate([numeric, one_hot], axis=1)`.
- **Layout conversion**: NHWC to NCHW when moving between TensorFlow and PyTorch conventions.
- **Padding variable length sequences** to a common length so they fit in one tensor.
- **Adding or removing the batch axis** at inference boundaries.
- **Splitting** a fused QKV projection into separate Q, K, and V tensors.

In [ ]:
# reshape, and the -1 placeholder that infers one axis for you.
a = np.arange(24)
print("original      :", a.shape)
print("reshape(4,6)  :", a.reshape(4, 6).shape)
print("reshape(2,3,4):", a.reshape(2, 3, 4).shape)
print("reshape(-1,4) :", a.reshape(-1, 4).shape, " NumPy computes the 6")
print("reshape(2,-1) :", a.reshape(2, -1).shape)

# The classic CNN flatten. Keep the batch axis, collapse everything else.
feat = rng.normal(size=(32, 64, 7, 7)).astype(np.float32)
flat = feat.reshape(feat.shape[0], -1)
print("\nfeature maps  :", feat.shape)
print("flattened     :", flat.shape, " ready for a dense layer")

# ravel returns a view when possible, flatten always copies.
print("\nravel shares memory  :", np.shares_memory(feat, feat.ravel()))
print("flatten shares memory:", np.shares_memory(feat, feat.flatten()))

In [ ]:
# transpose, swapaxes, moveaxis: three ways to say the same thing.
x = rng.normal(size=(2, 3, 4, 5)).astype(np.float32)
print("original            :", x.shape)
print("transpose(0,2,3,1)  :", x.transpose(0, 2, 3, 1).shape, " NCHW to NHWC")
print("swapaxes(1, 3)      :", x.swapaxes(1, 3).shape)
print("moveaxis(1, -1)     :", np.moveaxis(x, 1, -1).shape, " most readable")
print("\n.T on 2D is a transpose, on >2D it reverses ALL axes, which is rarely what you want:")
print("x.T shape           :", x.T.shape)

In [ ]:
# expand_dims and squeeze: adding and removing length 1 axes.
img = rng.normal(size=(3, 224, 224)).astype(np.float32)

batched = np.expand_dims(img, axis=0)     # same as img[None] or img[np.newaxis]
print("single image :", img.shape)
print("as batch     :", batched.shape)

pred = rng.normal(size=(1, 1, 10)).astype(np.float32)
print("\nmodel output :", pred.shape)
print("squeeze()    :", np.squeeze(pred).shape,          " removes ALL length 1 axes")
print("squeeze(0)   :", np.squeeze(pred, axis=0).shape,  " removes only axis 0, safer")
print("\nBare squeeze() on a batch of size 1 silently deletes your batch axis.")
print("Always name the axis in production code.")

In [ ]:
# concatenate versus stack. Learn this once.
a = np.zeros((2, 3), dtype=np.float32)
b = np.ones((2, 3), dtype=np.float32)

print("concatenate axis=0:", np.concatenate([a, b], axis=0).shape, " (4,3), joins rows, ndim stays 2")
print("concatenate axis=1:", np.concatenate([a, b], axis=1).shape, " (2,6), joins columns")
print("stack       axis=0:", np.stack([a, b], axis=0).shape,       " (2,2,3), NEW axis, ndim is now 3")
print("stack       axis=-1:", np.stack([a, b], axis=-1).shape,     " (2,3,2)")

# Data loader collation: individual samples become a batch with stack.
samples = [rng.normal(size=(3, 32, 32)).astype(np.float32) for _ in range(8)]
batch = np.stack(samples, axis=0)
print("\n8 samples of", samples[0].shape, "collated into", batch.shape)

# Feature assembly: numeric columns plus one hot columns, joined with concatenate.
numeric = rng.normal(size=(100, 5)).astype(np.float32)
categorical = np.eye(4, dtype=np.float32)[rng.integers(0, 4, size=100)]
X_full = np.concatenate([numeric, categorical], axis=1)
print("numeric", numeric.shape, "+ one hot", categorical.shape, "=", X_full.shape)

In [ ]:
# Padding variable length sequences into one rectangular batch.
sequences = [np.array([1, 2, 3]),
             np.array([4, 5, 6, 7, 8]),
             np.array([9])]
pad_id, max_len = 0, max(len(s) for s in sequences)

padded = np.full((len(sequences), max_len), pad_id, dtype=np.int32)
attention_mask = np.zeros((len(sequences), max_len), dtype=np.int32)
for i, s in enumerate(sequences):
    padded[i, :len(s)] = s
    attention_mask[i, :len(s)] = 1

print("padded ids:\n", padded)
print("attention mask:\n", attention_mask)
print("\nThe mask is not optional. Without it the model attends to padding")
print("and your pooled representations are diluted by meaningless tokens.")

# np.pad for images: reflect padding avoids the dark border that zero padding creates.
small_img = rng.random((4, 4)).astype(np.float32)
print("\nzero padded   :", np.pad(small_img, 2, mode="constant").shape)
print("reflect padded:", np.pad(small_img, 2, mode="reflect").shape)

In [ ]:
# split, array_split, and the fused QKV projection split.
data = np.arange(20)
print("split into 4 equal parts:", [p.tolist() for p in np.split(data, 4)])
print("array_split handles uneven sizes:",
      [len(p) for p in np.array_split(np.arange(23), 4)])

# Transformers compute Q, K, V in one matmul then split, because one big
# matmul is much faster than three small ones.
batch, seq, d_model = 2, 6, 12
qkv = rng.normal(size=(batch, seq, 3 * d_model)).astype(np.float32)
q, k, v = np.split(qkv, 3, axis=-1)
print("\nfused qkv:", qkv.shape, "-> q,k,v each", q.shape)

# repeat versus tile.
r = np.array([1, 2, 3])
print("\nrepeat(2):", np.repeat(r, 2), " each element repeated in place")
print("tile(2)  :", np.tile(r, 2),   " the whole array repeated")

In [ ]:
# A mini batch generator, the pattern every training loop uses.
def batches(X, y, batch_size, generator, shuffle=True):
    n = len(X)
    idx = generator.permutation(n) if shuffle else np.arange(n)
    for start in range(0, n, batch_size):
        sel = idx[start:start + batch_size]
        yield X[sel], y[sel]          # fancy indexing, so these are copies

Xb = rng.normal(size=(1000, 10)).astype(np.float32)
yb = rng.integers(0, 3, size=1000)
g = np.random.default_rng(0)

sizes = [len(xb) for xb, _ in batches(Xb, yb, 128, g)]
print("batch sizes:", sizes, " <- the last batch is smaller, handle it")
print("total samples covered:", sum(sizes))

---
# 14. Sliding windows and strides

## What it is

`strides` tells NumPy how many bytes to step to move one position along each axis. Because shape and strides are just metadata, NumPy can present the same buffer as overlapping windows **without copying anything**.

`np.lib.stride_tricks.sliding_window_view` is the safe, supported way to do this. `as_strided` is the raw version, and it will happily read past the end of your buffer and segfault or return garbage if you get the arithmetic wrong.

## Why production cares

Windowing is how you turn a time series into a supervised learning problem. Doing it with a Python loop that copies each window multiplies your memory by the window length. A strided view uses zero extra memory for the windows themselves.

## Where it shows up

- **Time series forecasting**: build `(n_windows, lookback, n_features)` inputs for an LSTM or a temporal transformer.
- **Rolling features**: moving averages, rolling standard deviation, rolling max for anomaly detection.
- **Audio framing**: cutting a waveform into overlapping frames before an FFT to build a spectrogram.
- **Patch extraction** for Vision Transformers: an image becomes a sequence of flattened patches.
- **Sequence labelling**: n gram context windows around each token.

In [ ]:
from numpy.lib.stride_tricks import sliding_window_view

series = np.arange(10, dtype=np.float32)
windows = sliding_window_view(series, window_shape=4)
print("series :", series)
print("windows:\n", windows)
print("shape  :", windows.shape, " (n - w + 1, w)")
print("copy made?", not np.shares_memory(series, windows), " <- it is a VIEW, zero extra memory")

In [ ]:
# Supervised windowing for forecasting: X is the lookback, y is the next value.
def make_supervised(series, lookback, horizon=1):
    w = sliding_window_view(series, lookback + horizon)
    X = w[:, :lookback]
    y = w[:, lookback:]
    return X, y

prices = np.cumsum(rng.normal(size=200)).astype(np.float32) + 100
X_seq, y_seq = make_supervised(prices, lookback=24, horizon=1)
print("series length:", len(prices))
print("X:", X_seq.shape, " y:", y_seq.shape)
print("first window:", np.round(X_seq[0][:5], 2), "... target:", np.round(y_seq[0], 2))

# Multivariate: (timesteps, features) into (n_windows, lookback, features).
multi = rng.normal(size=(200, 5)).astype(np.float32)
mw = sliding_window_view(multi, window_shape=24, axis=0)   # (177, 5, 24)
mw = np.moveaxis(mw, -1, 1)                                # (177, 24, 5) for an LSTM
print("\nmultivariate windows:", mw.shape, " (batch, timesteps, features)")

In [ ]:
# Rolling statistics, vectorized.
signal = rng.normal(size=1000).astype(np.float32)
w = 20
rolled = sliding_window_view(signal, w)

roll_mean = rolled.mean(axis=1)
roll_std  = rolled.std(axis=1)
roll_max  = rolled.max(axis=1)
print("rolling stats shape:", roll_mean.shape)

# Anomaly flagging with a rolling z score, a standard monitoring rule.
z = (signal[w-1:] - roll_mean) / (roll_std + 1e-8)
anomalies = np.flatnonzero(np.abs(z) > 3)
print("anomalies detected:", len(anomalies), "at positions", anomalies[:10])

In [ ]:
# Patchify an image for a Vision Transformer.
img = rng.random((1, 3, 32, 32)).astype(np.float32)     # (batch, C, H, W)
p = 8                                                    # patch size

patches = sliding_window_view(img, (p, p), axis=(2, 3))[:, :, ::p, ::p]
print("windowed then strided:", patches.shape, " (B, C, H/p, W/p, p, p)")

B, C, gh, gw, ph, pw = patches.shape
tokens = patches.transpose(0, 2, 3, 1, 4, 5).reshape(B, gh * gw, C * ph * pw)
print("as a token sequence  :", tokens.shape, " (batch, num_patches, patch_dim)")
print(f"  {gh}x{gw} = {gh*gw} patches, each {C}x{p}x{p} = {C*p*p} numbers flattened")
print("\nThis is literally the first layer of a ViT, before the linear projection.")

In [ ]:
# Audio framing before an FFT, the first step of a spectrogram.
sr = 16_000
audio = rng.normal(size=sr).astype(np.float32)      # one second
frame_len, hop = 400, 160                            # 25 ms frames, 10 ms hop

frames = sliding_window_view(audio, frame_len)[::hop]
print("audio  :", audio.shape)
print("frames :", frames.shape, " (n_frames, frame_length)")

windowed = frames * np.hanning(frame_len).astype(np.float32)   # reduce spectral leakage
spectrum = np.abs(np.fft.rfft(windowed, axis=1))
print("spectrogram:", spectrum.shape, " (n_frames, n_freq_bins)")
print("This 2D array is what you feed to a CNN for speech or audio classification.")

---
# 15. Sorting, searching, and set operations

## What it is

Ordering elements (`sort`, `argsort`), partial ordering for top-k (`argpartition`), binary search on sorted data (`searchsorted`), and set style operations (`unique`, `isin`, `intersect1d`).

The `arg` prefix means "return the indices, not the values". Indices are what you actually want, because they let you reorder several parallel arrays consistently (scores, ids, and metadata all at once).

## Why production cares

Ranking is the core of search, recommendation, and retrieval. `argsort` is O(n log n) and sorts everything. `argpartition` is O(n) and only guarantees that the top k are in the first k positions, which is all a top-k query needs. At 10 million candidates that difference is the difference between meeting and missing a latency budget.

## Where it shows up

- **Top-k retrieval** in recommenders and vector search.
- **Ranking metrics**: NDCG, MAP, precision at k, all built on sorted scores.
- **Percentiles and quantile binning** of features.
- **Vocabulary building**: `np.unique` gives sorted unique tokens plus counts.
- **Label encoding**: mapping arbitrary category values to contiguous integer ids.
- **Class weights** from `np.bincount`.
- **Filtering by an allow list** with `np.isin`.
- **Weighted sampling** with `searchsorted` on a cumulative distribution.

In [ ]:
scores = np.array([0.2, 0.9, 0.5, 0.1, 0.8, 0.35], dtype=np.float32)
item_ids = np.array([101, 102, 103, 104, 105, 106])

order = np.argsort(-scores)                    # negate for descending
print("scores sorted desc :", np.round(scores[order], 2))
print("item ids in order  :", item_ids[order])
print("\nUsing the same index array keeps scores, ids, and any metadata aligned.")

k = 3
top_k_unordered = np.argpartition(-scores, k)[:k]              # O(n), unordered within k
top_k = top_k_unordered[np.argsort(-scores[top_k_unordered])]  # sort only those k
print("\ntop 3 item ids     :", item_ids[top_k])
print("top 3 scores       :", np.round(scores[top_k], 2))

In [ ]:
# Top-k for a whole batch of queries at once, with take_along_axis.
sims = rng.random((4, 1000)).astype(np.float32)     # 4 queries against 1000 items
k = 5

idx = np.argpartition(-sims, k, axis=1)[:, :k]
vals = np.take_along_axis(sims, idx, axis=1)
order = np.argsort(-vals, axis=1)
top_idx = np.take_along_axis(idx, order, axis=1)
top_val = np.take_along_axis(vals, order, axis=1)

print("top item indices per query:\n", top_idx)
print("top scores per query:\n", np.round(top_val, 3))
print("\nNo Python loop over queries. This is how a recommender serves a batch.")

In [ ]:
# unique: vocabulary building and label encoding in one call.
raw_labels = np.array(["cat", "dog", "cat", "bird", "dog", "cat"])

classes, encoded, counts = np.unique(raw_labels, return_inverse=True, return_counts=True)
print("classes        :", classes)
print("encoded labels :", encoded, " <- ready for a loss function")
print("counts         :", counts)
print("class weights  :", np.round(counts.sum() / (len(classes) * counts), 3))
print("\n`classes` is the artifact you must save. Decoding predictions at serving")
print("time uses classes[pred_id]. Recomputing it on new data reorders everything.")

In [ ]:
# bincount, isin, and quantile binning.
y = rng.integers(0, 5, size=1000)
print("class counts   :", np.bincount(y))

allowed = np.array([1, 3])
keep = np.isin(y, allowed)
print("kept by allow list:", keep.sum())

# Quantile binning: equal frequency buckets, a standard tabular feature transform.
feature = rng.exponential(scale=2.0, size=1000).astype(np.float32)
edges = np.quantile(feature, [0.2, 0.4, 0.6, 0.8])
bins = np.searchsorted(edges, feature)     # 0..4
print("\nbin edges      :", np.round(edges, 3))
print("bin counts     :", np.bincount(bins))
print("Save `edges` with the model. Recomputing bins on production data shifts them.")

In [ ]:
# searchsorted for weighted sampling, used in negative sampling and replay buffers.
weights = np.array([0.1, 0.5, 0.3, 0.1])
cdf = np.cumsum(weights)
g = np.random.default_rng(0)
draws = np.searchsorted(cdf, g.random(10_000))
print("empirical frequencies:", np.round(np.bincount(draws, minlength=4) / 10_000, 3))
print("target weights       :", weights)
print("\nOne cumsum plus a binary search: O(log n) per draw, fully vectorized.")

---
# 16. Saving, loading, and data bigger than RAM

## What it is

- `np.save` / `np.load` write a single array to `.npy`, preserving dtype and shape exactly.
- `np.savez` / `np.savez_compressed` write several named arrays to one `.npz`.
- `np.memmap` and `np.load(..., mmap_mode="r")` map a file on disk into your address space so you can slice a 200 GB array while only paging in what you touch.

## Why production cares

CSV is a terrible format for numerical data. It stores floats as text, so it is 2 to 3 times larger, loses precision, and parses 10 to 100 times slower. Switching an intermediate artifact from CSV to `.npy` is often the cheapest performance win available.

Memory mapping is what lets a training job read a dataset larger than the machine's RAM, with the operating system handling the caching.

## Where it shows up

- Caching preprocessed features so you do not repeat expensive transforms every epoch.
- Storing an embedding index for a retrieval service.
- Shipping scaler statistics, vocabularies, and class lists alongside a model.
- Large image and audio datasets stored as one flat memmapped file with an index.

## Security note

`np.load` with `allow_pickle=True` executes arbitrary code from the file. Never enable it for data you did not create. The default is `False` and it should stay that way.

In [ ]:
import os, tempfile
tmpdir = tempfile.mkdtemp()

X = rng.normal(size=(10_000, 50)).astype(np.float32)
y = rng.integers(0, 3, size=10_000)

npy_path = os.path.join(tmpdir, "features.npy")
np.save(npy_path, X)
X_loaded = np.load(npy_path)
print("round trip exact:", np.array_equal(X, X_loaded), " dtype preserved:", X_loaded.dtype)

csv_path = os.path.join(tmpdir, "features.csv")
np.savetxt(csv_path, X[:1000], delimiter=",")
print(f"\n.npy  {os.path.getsize(npy_path)/1e6:6.2f} MB for 10000 rows")
print(f".csv  {os.path.getsize(csv_path)/1e6:6.2f} MB for only 1000 rows")

In [ ]:
# Bundle every artifact a model needs to be served.
bundle = os.path.join(tmpdir, "model_artifacts.npz")
np.savez_compressed(
    bundle,
    weights=rng.normal(size=(50, 3)).astype(np.float32),
    bias=np.zeros(3, dtype=np.float32),
    feature_mean=X.mean(axis=0),
    feature_std=X.std(axis=0),
    classes=np.array(["low", "medium", "high"]),
)

art = np.load(bundle)
print("stored arrays:", art.files)
print("feature_mean shape:", art["feature_mean"].shape)
print("classes           :", art["classes"])
print("\nA model without its preprocessing statistics is not a deployable model.")

In [ ]:
# Memory mapping: work with a file larger than RAM.
mm_path = os.path.join(tmpdir, "big.dat")
shape = (100_000, 128)

# Write in chunks, never holding the whole array in memory.
mm = np.memmap(mm_path, dtype=np.float32, mode="w+", shape=shape)
chunk = 10_000
for start in range(0, shape[0], chunk):
    mm[start:start + chunk] = rng.normal(size=(chunk, shape[1])).astype(np.float32)
mm.flush()
del mm

# Read back lazily: only the pages you touch are loaded.
mm_read = np.memmap(mm_path, dtype=np.float32, mode="r", shape=shape)
print("on disk :", os.path.getsize(mm_path) / 1e6, "MB")
print("slice   :", mm_read[500:510, :4].shape, " only these pages were read")

# Chunked reduction over the whole file without loading it.
total, count = np.zeros(shape[1], dtype=np.float64), 0
for start in range(0, shape[0], chunk):
    block = np.asarray(mm_read[start:start + chunk], dtype=np.float64)
    total += block.sum(axis=0)
    count += len(block)
print("column means (first 4):", np.round(total[:4] / count, 4))
del mm_read

In [ ]:
# Zero copy handoff to a deep learning framework.
arr = np.ascontiguousarray(rng.normal(size=(4, 3, 32, 32)).astype(np.float32))
print("C contiguous:", arr.flags["C_CONTIGUOUS"], " dtype:", arr.dtype)
# In PyTorch:
#     t = torch.from_numpy(arr)        # shares memory, no copy, instant
#     t = torch.as_tensor(arr)         # same when dtype and layout already match
#
# Requirements for the zero copy path:
#     1. The array is C contiguous. Call np.ascontiguousarray if a transpose broke it.
#     2. The dtype matches what the framework wants, normally float32.
#     3. The array is writable and not a strided trick view.
#
# Fail any of these and the framework silently copies, which is a per batch cost
# you pay on every step of every epoch.
print("array is ready for a zero copy handoff:",
      arr.flags["C_CONTIGUOUS"] and arr.dtype == np.float32 and arr.flags["WRITEABLE"])

---
# 17. Performance engineering

## What it is

Getting the most out of vectorized code once the obvious loops are gone. The remaining levers are memory layout, avoiding copies, reusing buffers, and choosing the operation with the better implementation behind it.

## The rules, in order of impact

1. **Vectorize first.** Everything else is a rounding error next to removing a Python loop.
2. **Use float32 unless you have a reason not to.** Half the memory means half the memory bandwidth, and most numerical code is bandwidth bound.
3. **Keep arrays C contiguous.** A transposed array has strides that defeat the CPU cache. `np.ascontiguousarray` costs one copy and can pay for itself many times over.
4. **Never grow an array in a loop.** `np.append` and `np.concatenate` inside a loop reallocate and copy everything, every iteration, giving quadratic behaviour. Collect into a Python list and stack once, or preallocate.
5. **Reuse buffers** with `out=` and in place operators in hot paths.
6. **Reach for a matmul.** BLAS is more optimized than anything you will write.
7. **Process in chunks** when an intermediate would be huge, and know your peak memory, not just your input size.
8. **Know when to leave NumPy.** NumPy is single threaded for elementwise work. If you are still too slow after all of the above, the answer is Numba, PyTorch, JAX, or a GPU, not more NumPy tricks.

In [ ]:
# Rule 4 in numbers: growing versus preallocating versus list plus stack.
n = 3000

t0 = time.perf_counter()
grown = np.array([], dtype=np.float32)
for i in range(n):
    grown = np.append(grown, np.float32(i))          # reallocates EVERY time
t_grow = time.perf_counter() - t0

t0 = time.perf_counter()
pre = np.empty(n, dtype=np.float32)
for i in range(n):
    pre[i] = i
t_pre = time.perf_counter() - t0

t0 = time.perf_counter()
acc = [np.float32(i) for i in range(n)]
stacked = np.array(acc, dtype=np.float32)
t_list = time.perf_counter() - t0

print(f"np.append in loop : {t_grow*1000:8.2f} ms   O(n^2)")
print(f"preallocate       : {t_pre*1000:8.2f} ms   O(n)")
print(f"list then array   : {t_list*1000:8.2f} ms   O(n)")
print(f"append is {t_grow/max(t_pre,1e-9):.0f}x slower, and the gap grows with n")

In [ ]:
# Rule 3: memory layout changes speed even when the maths is identical.
A = np.ascontiguousarray(rng.normal(size=(2000, 2000)).astype(np.float32))
A_t = np.asfortranarray(A)          # same values, column major layout

t0 = time.perf_counter(); s1 = A.sum(axis=1);   t_c = time.perf_counter() - t0
t0 = time.perf_counter(); s2 = A_t.sum(axis=1); t_f = time.perf_counter() - t0

print(f"row sums on C ordered      : {t_c*1000:7.2f} ms")
print(f"row sums on Fortran ordered: {t_f*1000:7.2f} ms")
print("Summing along the fast (last) axis walks contiguous memory and hits cache.")
print("same result:", np.allclose(s1, s2))

In [ ]:
# Rule 7: peak memory is about intermediates, not inputs.
n = 2_000_000
x = rng.random(n).astype(np.float32)

# Each temporary here is another 8 MB allocation.
def many_temporaries(x):
    return np.sqrt(np.abs(x * 2.0 - 1.0)) + 3.0

# In place chain: one buffer, reused.
def few_temporaries(x):
    out = x * 2.0
    out -= 1.0
    np.abs(out, out=out)
    np.sqrt(out, out=out)
    out += 3.0
    return out

t0 = time.perf_counter(); r1 = many_temporaries(x); t1 = time.perf_counter() - t0
t0 = time.perf_counter(); r2 = few_temporaries(x);  t2 = time.perf_counter() - t0
print(f"with temporaries : {t1*1000:7.2f} ms")
print(f"in place chain   : {t2*1000:7.2f} ms")
print("same result:", np.allclose(r1, r2))
print(f"\nEach temporary of this array is {x.nbytes/1e6:.0f} MB. Four of them at once")
print("is what turns a comfortable job into an out of memory crash.")

In [ ]:
# Rule 6: express it as a matmul when you can.
n, d = 2000, 256
A = rng.normal(size=(n, d)).astype(np.float32)
B = rng.normal(size=(d, n)).astype(np.float32)

t0 = time.perf_counter(); C = A @ B; t_mm = time.perf_counter() - t0
flops = 2 * n * n * d
print(f"matmul {A.shape} x {B.shape}: {t_mm*1000:.1f} ms, "
      f"{flops/t_mm/1e9:.1f} GFLOP/s")
print("BLAS is multithreaded, cache blocked, and SIMD vectorized.")
print("A hand written triple loop for the same work would take minutes.")

---
# 18. Testing numerical code

## What it is

Unit testing code that produces floating point numbers. Exact equality almost never holds, so you assert closeness, shapes, dtypes, and invariants instead.

## Why production cares

Numerical bugs are silent. A shape bug that broadcasts, an axis bug that averages the wrong direction, a leaked test statistic in a scaler: none of these raise. They just make your model slightly worse, and you find out weeks later from a business metric.

## What to assert

- **Shape and dtype** of every output. Cheap, and it catches most bugs.
- **Closeness**, with `np.allclose` or `np.testing.assert_allclose`, with a tolerance appropriate to the dtype.
- **Invariants**: probabilities sum to 1 and lie in [0, 1], a normalized array has mean 0 and standard deviation 1, a distance matrix is symmetric with a zero diagonal, a covariance matrix is positive semi definite.
- **Finiteness**: no NaN, no Inf, anywhere in the output.
- **Known cases**: hand computed small examples where you know the answer.
- **Gradient checks**: compare an analytic gradient against a finite difference approximation.

In [ ]:
# Assert closeness, not equality.
a = np.array([0.1, 0.2, 0.3])
b = np.array([0.1, 0.2, 0.1 + 0.2])
print("exact equality :", np.array_equal(a, b))
print("allclose       :", np.allclose(a, b))

# assert_allclose gives a detailed diff when it fails, which is what you want in CI.
np.testing.assert_allclose(a, b, rtol=1e-6)
print("assert_allclose passed")

# Tolerances should match the dtype.
print("\nfloat32 eps:", np.finfo(np.float32).eps, " use rtol around 1e-5")
print("float64 eps:", np.finfo(np.float64).eps, " use rtol around 1e-9")

In [ ]:
# A realistic test suite for a preprocessing function.
def standardize(X, mean=None, std=None, eps=1e-8):
    X = np.asarray(X, dtype=np.float32)
    if mean is None:
        mean = X.mean(axis=0)
    if std is None:
        std = X.std(axis=0)
    return (X - mean) / (std + eps), mean, std

def test_standardize():
    g = np.random.default_rng(0)
    X = g.normal(loc=10, scale=3, size=(100, 5)).astype(np.float32)

    Xs, mu, sd = standardize(X)

    assert Xs.shape == X.shape, "shape must be preserved"
    assert Xs.dtype == np.float32, "dtype must not be upcast"
    np.testing.assert_allclose(Xs.mean(axis=0), 0, atol=1e-5)
    np.testing.assert_allclose(Xs.std(axis=0), 1, atol=1e-4)
    assert np.isfinite(Xs).all(), "no nan or inf"

    # Applying stored statistics must be deterministic and identical.
    Xs2, _, _ = standardize(X, mean=mu, std=sd)
    np.testing.assert_allclose(Xs, Xs2)

    # A constant column must not produce NaN.
    Xc = np.ones((10, 2), dtype=np.float32)
    out, _, _ = standardize(Xc)
    assert np.isfinite(out).all(), "constant column must not divide by zero"
    return "all assertions passed"

print(test_standardize())

In [ ]:
# Gradient check: the standard way to validate a hand written backward pass.
def f(w):
    return float((w ** 2).sum() + 3 * w.sum())

def analytic_grad(w):
    return 2 * w + 3

w = np.array([1.0, -2.0, 0.5], dtype=np.float64)   # float64 for the finite difference
eps = 1e-6
numeric = np.empty_like(w)
for i in range(len(w)):
    wp, wm = w.copy(), w.copy()
    wp[i] += eps
    wm[i] -= eps
    numeric[i] = (f(wp) - f(wm)) / (2 * eps)       # central difference

print("analytic:", analytic_grad(w))
print("numeric :", np.round(numeric, 6))
print("max abs difference:", np.abs(analytic_grad(w) - numeric).max())
np.testing.assert_allclose(analytic_grad(w), numeric, rtol=1e-5)
print("gradient check passed")

---
# 19. Production pitfalls checklist

Print this. Every item below has caused a real incident somewhere.

| # | Pitfall | Symptom | Fix |
|---|---------|---------|-----|
| 1 | Slicing returns a view | Caller's array mutates unexpectedly | `.copy()` when you intend to own the data |
| 2 | `(n,)` vs `(n,1)` broadcasting | Loss silently becomes a matrix | Assert shapes at function boundaries |
| 3 | Wrong `axis` in a reduction | Metrics subtly wrong, no error | Remember: `axis=k` means axis k disappears |
| 4 | float64 everywhere | Out of memory, slow training | Cast to float32 at the pipeline boundary |
| 5 | Integer overflow | Negative counts, wrong sums | Use int64, or pass `dtype=` to `sum` |
| 6 | `exp` overflow in softmax | inf then NaN loss | Subtract the max before `exp` |
| 7 | Division by zero std | NaN after normalization | Add an epsilon, always |
| 8 | Refitting the scaler on test | Optimistic offline metrics, bad production model | Fit on train, apply everywhere |
| 9 | Shuffling time series | Massive leakage, unreproducible in production | Split by time |
| 10 | Global `np.random.seed` | Non reproducible runs, identical augmentations across workers | Use `default_rng` per component |
| 11 | `np.append` in a loop | Quadratic slowdown | Preallocate, or list then `np.stack` |
| 12 | `np.vectorize` for speed | No speedup at all | Use real array expressions |
| 13 | Bare `squeeze()` | Batch axis vanishes when batch size is 1 | `squeeze(axis=...)` |
| 14 | HWC vs CHW mix up | Garbage predictions, no error | Convert explicitly and assert the shape |
| 15 | Forgetting the padding mask | Pooled embeddings diluted by pad tokens | Mask before pooling |
| 16 | `and` / `or` on arrays | `ValueError: truth value ambiguous` | Use `&`, `|`, `~` with parentheses |
| 17 | Huge broadcast intermediate | Out of memory on a distance matrix | Use the matmul identity, or chunk |
| 18 | `np.load(allow_pickle=True)` on untrusted data | Arbitrary code execution | Keep the default `False` |
| 19 | Non contiguous array to a framework | Hidden per batch copy | `np.ascontiguousarray` once |
| 20 | Comparing floats with `==` | Flaky tests | `np.isclose` / `assert_allclose` |

In [ ]:
# A defensive helper worth copying into your own projects.
def validate_batch(X, y=None, expected_features=None, name="batch"):
    X = np.asarray(X)
    problems = []
    if X.ndim != 2:
        problems.append(f"expected 2D (n_samples, n_features), got {X.ndim}D {X.shape}")
    if expected_features is not None and X.ndim == 2 and X.shape[1] != expected_features:
        problems.append(f"expected {expected_features} features, got {X.shape[1]}")
    if not np.issubdtype(X.dtype, np.floating):
        problems.append(f"expected a float dtype, got {X.dtype}")
    if X.dtype == np.float64:
        problems.append("float64 detected, cast to float32 to halve memory")
    if not np.isfinite(X).all():
        problems.append(f"{int((~np.isfinite(X)).sum())} non finite values")
    if y is not None:
        y = np.asarray(y)
        if len(y) != len(X):
            problems.append(f"X has {len(X)} rows but y has {len(y)}")
    return problems

good = rng.normal(size=(100, 8)).astype(np.float32)
bad = rng.normal(size=(100, 5))          # float64, wrong feature count
bad[3, 2] = np.nan

print("good batch:", validate_batch(good, expected_features=8) or "no problems")
print("\nbad batch:")
for p in validate_batch(bad, expected_features=8):
    print("  -", p)

---
# 20. End to end mini project: a complete pipeline in pure NumPy

Everything above, assembled into one working system. No scikit-learn, no PyTorch. The point is that once you can express these operations in NumPy, you understand exactly what those libraries do for you.

**The pipeline**

1. Generate a synthetic three class dataset with realistic problems: correlated features, outliers, and missing values.
2. Split reproducibly.
3. Clean: impute missing values and clip outliers, using **training statistics only**.
4. Standardize with training statistics only.
5. Reduce dimensionality with PCA.
6. Train a softmax regression classifier with mini batch gradient descent.
7. Evaluate with accuracy, a confusion matrix, per class F1, and a bootstrap confidence interval.
8. Save every artifact required to serve the model.
9. Run a single prediction the way a serving endpoint would.

In [ ]:
# Step 1: synthetic data with realistic defects.
def make_dataset(n=3000, d=12, n_classes=3, seed=0):
    g = np.random.default_rng(seed)
    centers = g.normal(scale=2.0, size=(n_classes, d))
    y = g.integers(0, n_classes, size=n)
    X = centers[y] + g.normal(scale=1.5, size=(n, d))

    X[:, 3] = X[:, 0] * 0.9 + g.normal(scale=0.1, size=n)     # correlated feature
    out_idx = g.choice(n, size=n // 50, replace=False)
    X[out_idx] *= 12.0                                        # outliers
    nan_idx = g.choice(n, size=n // 40, replace=False)
    X[nan_idx, g.integers(0, d, size=len(nan_idx))] = np.nan  # missing values
    return X.astype(np.float32), y.astype(np.int64)

X, y = make_dataset()
print("X:", X.shape, X.dtype, " y:", y.shape, y.dtype)
print("class counts :", np.bincount(y))
print("missing cells:", int(np.isnan(X).sum()))
print("memory       :", f"{X.nbytes/1e6:.2f} MB")

In [ ]:
# Step 2: reproducible split.
def split_indices(n, seed=42, train=0.7, val=0.15):
    g = np.random.default_rng(seed)
    perm = g.permutation(n)
    a, b = int(train * n), int((train + val) * n)
    return perm[:a], perm[a:b], perm[b:]

i_tr, i_va, i_te = split_indices(len(X))
X_tr, y_tr = X[i_tr], y[i_tr]
X_va, y_va = X[i_va], y[i_va]
X_te, y_te = X[i_te], y[i_te]
print("train / val / test:", len(i_tr), len(i_va), len(i_te))
print("no overlap:", len(np.intersect1d(i_tr, i_te)) == 0)

In [ ]:
# Step 3 and 4: fit the preprocessing on TRAIN only, then apply to all splits.
class Preprocessor:
    def fit(self, X):
        self.impute_ = np.nanmedian(X, axis=0).astype(np.float32)
        Xi = np.where(np.isnan(X), self.impute_, X)
        self.lo_, self.hi_ = np.percentile(Xi, [1, 99], axis=0).astype(np.float32)
        Xc = np.clip(Xi, self.lo_, self.hi_)
        self.mean_ = Xc.mean(axis=0)
        self.std_ = Xc.std(axis=0) + 1e-8
        return self

    def transform(self, X):
        X = np.asarray(X, dtype=np.float32)
        X = np.where(np.isnan(X), self.impute_, X)
        X = np.clip(X, self.lo_, self.hi_)
        return (X - self.mean_) / self.std_

prep = Preprocessor().fit(X_tr)
X_tr_p, X_va_p, X_te_p = prep.transform(X_tr), prep.transform(X_va), prep.transform(X_te)

print("train mean after preprocessing:", np.round(X_tr_p.mean(axis=0)[:4], 5))
print("train std  after preprocessing:", np.round(X_tr_p.std(axis=0)[:4], 5))
print("test  mean after preprocessing:", np.round(X_te_p.mean(axis=0)[:4], 4),
      " (not exactly 0, which is correct)")
print("any NaN left:", np.isnan(X_tr_p).any() or np.isnan(X_te_p).any())

In [ ]:
# Step 5: PCA, fitted on train only.
class PCA:
    def __init__(self, n_components):
        self.k = n_components

    def fit(self, X):
        self.mean_ = X.mean(axis=0)
        U, S, Vt = np.linalg.svd(X - self.mean_, full_matrices=False)
        self.components_ = Vt[:self.k]
        self.explained_ = (S ** 2 / (S ** 2).sum())[:self.k]
        return self

    def transform(self, X):
        return (X - self.mean_) @ self.components_.T

pca = PCA(n_components=8).fit(X_tr_p)
X_tr_r, X_va_r, X_te_r = pca.transform(X_tr_p), pca.transform(X_va_p), pca.transform(X_te_p)

print("reduced shapes:", X_tr_r.shape, X_va_r.shape, X_te_r.shape)
print("explained variance:", np.round(pca.explained_, 4))
print("cumulative        :", np.round(np.cumsum(pca.explained_), 4))

In [ ]:
# Step 6: softmax regression trained with mini batch gradient descent.
def log_softmax(z, axis=-1):
    zm = z - z.max(axis=axis, keepdims=True)
    return zm - np.log(np.exp(zm).sum(axis=axis, keepdims=True))

def train_softmax(X, y, Xv, yv, n_classes, epochs=40, bs=128, lr=0.2, l2=1e-4, seed=0):
    g = np.random.default_rng(seed)
    n, d = X.shape
    W = (g.normal(size=(d, n_classes)) * np.sqrt(2.0 / d)).astype(np.float32)
    b = np.zeros(n_classes, dtype=np.float32)
    history = []

    for epoch in range(epochs):
        perm = g.permutation(n)
        for start in range(0, n, bs):
            sel = perm[start:start + bs]
            xb, yb = X[sel], y[sel]
            m = len(xb)

            logits = xb @ W + b
            probs = np.exp(log_softmax(logits))

            # Gradient of mean cross entropy: (softmax - onehot) / batch_size
            grad_logits = probs
            grad_logits[np.arange(m), yb] -= 1.0
            grad_logits /= m

            gW = xb.T @ grad_logits + l2 * W
            gb = grad_logits.sum(axis=0)
            W -= lr * gW
            b -= lr * gb

        tr_loss = -log_softmax(X @ W + b)[np.arange(n), y].mean()
        va_pred = (Xv @ W + b).argmax(axis=1)
        history.append((epoch, float(tr_loss), float((va_pred == yv).mean())))
    return W, b, history

W, b, hist = train_softmax(X_tr_r, y_tr, X_va_r, y_va, n_classes=3)

print("epoch  train_loss  val_acc")
for e, l, a in hist[::8] + [hist[-1]]:
    print(f"{e:5d}  {l:10.4f}  {a:7.4f}")

In [ ]:
# Step 7: evaluation.
def evaluate(X, y, W, b, n_classes=3):
    logits = X @ W + b
    pred = logits.argmax(axis=1)
    cm = np.zeros((n_classes, n_classes), dtype=np.int64)
    np.add.at(cm, (y, pred), 1)

    tp = np.diag(cm).astype(np.float64)
    precision = tp / (cm.sum(axis=0) + 1e-12)
    recall = tp / (cm.sum(axis=1) + 1e-12)
    f1 = 2 * precision * recall / (precision + recall + 1e-12)
    return {
        "accuracy": float((pred == y).mean()),
        "confusion": cm,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "macro_f1": float(f1.mean()),
        "pred": pred,
    }

res = evaluate(X_te_r, y_te, W, b)
print("test accuracy :", round(res["accuracy"], 4))
print("macro f1      :", round(res["macro_f1"], 4))
print("\nconfusion matrix (rows true, cols predicted):\n", res["confusion"])
print("\nper class precision:", np.round(res["precision"], 3))
print("per class recall   :", np.round(res["recall"], 3))
print("per class f1       :", np.round(res["f1"], 3))

In [ ]:
# Bootstrap confidence interval on test accuracy: is the number trustworthy?
g = np.random.default_rng(11)
correct = (res["pred"] == y_te)
idx = g.integers(0, len(correct), size=(2000, len(correct)))
boot = correct[idx].mean(axis=1)
lo, hi = np.percentile(boot, [2.5, 97.5])
print(f"test accuracy : {res['accuracy']:.4f}")
print(f"95% CI        : [{lo:.4f}, {hi:.4f}]  width {hi-lo:.4f}")
print(f"test set size : {len(y_te)}")
print("\nA smaller test set gives a wider interval. Report the interval, not just the point.")

In [ ]:
# Step 8: save every artifact needed to serve the model.
artifact_path = os.path.join(tmpdir, "pipeline.npz")
np.savez_compressed(
    artifact_path,
    impute=prep.impute_, clip_lo=prep.lo_, clip_hi=prep.hi_,
    mean=prep.mean_, std=prep.std_,
    pca_mean=pca.mean_, pca_components=pca.components_,
    W=W, b=b,
    class_names=np.array(["class_0", "class_1", "class_2"]),
    input_dim=np.array([X.shape[1]]),
)
print("saved:", os.path.basename(artifact_path),
      f"({os.path.getsize(artifact_path)/1e3:.1f} KB)")
print("contents:", np.load(artifact_path).files)
print("\nThe model weights alone are useless. Preprocessing statistics, PCA")
print("components, and the class name ordering are all part of the model.")

In [ ]:
# Step 9: serving. Load the artifacts and predict for a single request.
art = np.load(artifact_path)

def predict_one(raw_features, art):
    x = np.asarray(raw_features, dtype=np.float32).reshape(1, -1)
    assert x.shape[1] == int(art["input_dim"][0]), \
        f"expected {int(art['input_dim'][0])} features, got {x.shape[1]}"

    x = np.where(np.isnan(x), art["impute"], x)
    x = np.clip(x, art["clip_lo"], art["clip_hi"])
    x = (x - art["mean"]) / art["std"]
    x = (x - art["pca_mean"]) @ art["pca_components"].T

    logits = x @ art["W"] + art["b"]
    z = logits - logits.max(axis=1, keepdims=True)
    probs = np.exp(z) / np.exp(z).sum(axis=1, keepdims=True)
    k = int(probs.argmax())
    return {
        "label": str(art["class_names"][k]),
        "confidence": float(probs[0, k]),
        "probabilities": dict(zip(art["class_names"].tolist(), np.round(probs[0], 4).tolist())),
    }

sample = X_te[0].copy()
sample[2] = np.nan                      # a missing field in the request, handled
out = predict_one(sample, art)
print("prediction   :", out["label"])
print("confidence   :", round(out["confidence"], 4))
print("probabilities:", out["probabilities"])
print("true label   :", f"class_{y_te[0]}")

try:
    predict_one(np.zeros(5), art)
except AssertionError as e:
    print("\nvalidation caught a malformed request:", e)

---
# 21. Exercises

Work through these without looking at the solutions. Each maps to a real production task.

### Beginner

1. Build a `(64, 3, 32, 32)` uint8 batch of random images. Report its memory in MB, then convert to float32 in `[0, 1]` and report the new memory. How many times larger is it?
2. Given `X` of shape `(1000, 20)`, compute the mean and standard deviation **per feature**. Confirm the result has shape `(20,)` and explain why `axis=0` and not `axis=1`.
3. Given `logits` of shape `(32, 10)`, produce the predicted class for each sample and the confidence of each prediction, with no loop.
4. Take a 1D array of 100 daily prices and produce the 7 day moving average using `sliding_window_view`.

### Intermediate

5. Write `train_test_split(X, y, test_size, seed)` that is stratified: each class keeps its original proportion in both splits. Use `np.flatnonzero` and `permutation`.
6. Implement `cosine_similarity_matrix(A, B)` for `A` of shape `(n, d)` and `B` of shape `(m, d)`, returning `(n, m)`, using exactly one matmul. Handle zero norm rows without producing NaN.
7. Given `(batch, seq, dim)` token embeddings and a `(batch, seq)` binary padding mask, compute masked mean pooling, masked max pooling, and the count of real tokens per sequence.
8. Write `top_k_per_row(scores, k)` returning both indices and values, sorted descending, using `argpartition` rather than a full sort. Verify it against `argsort` on a small case.

### Advanced

9. Implement batched multi head attention for `(batch, seq, d_model)` input with `h` heads: project to Q, K, V, split into heads, apply a causal mask, use a numerically stable softmax, and merge the heads back. Assert the output shape equals the input shape.
10. Implement a memory efficient `pairwise_distances(A, B, chunk_size)` that never allocates more than `chunk_size * len(B)` floats at once, and confirm it matches the naive broadcasting version.
11. Write a `Standardizer` class with `fit`, `transform`, `save`, and `load`, plus a test suite asserting: shape preservation, dtype preservation, zero mean and unit variance on train, no NaN on a constant column, and identical results after a save and load round trip.
12. Profile three implementations of a rolling standard deviation over 1 million points (Python loop, `sliding_window_view`, and a cumulative sums approach) and explain the accuracy versus speed tradeoff of the cumulative sums version.

### Hints

- Exercise 5: `np.flatnonzero(y == c)` gives the positions for class `c`.
- Exercise 6: normalize both matrices first, then `A_norm @ B_norm.T`.
- Exercise 9: reshape `(B, S, d_model)` into `(B, S, h, d_head)`, then `transpose(0, 2, 1, 3)`.
- Exercise 10: loop over chunks of `A`, and use the `||a||^2 + ||b||^2 - 2ab` identity inside each chunk.
- Exercise 12: the cumulative sums approach uses `E[x^2] - E[x]^2`, which is fast and numerically fragile. Section 11 explains why.

---
# Quick reference

## Shape conventions

| Shape | Meaning |
|-------|---------|
| `()` | a scalar: loss, metric, hyperparameter |
| `(d,)` | one feature vector, one embedding, one time series |
| `(n, d)` | design matrix, batch of embeddings, weight matrix, grayscale image |
| `(n, t, d)` | batch of sequences, or one RGB image as `(H, W, C)` |
| `(n, c, h, w)` | batch of images, NCHW, PyTorch |
| `(n, h, w, c)` | batch of images, NHWC, TensorFlow |
| `(o, i, kh, kw)` | convolution kernel weights |
| `(n, heads, t, dh)` | multi head attention state |
| `(n, t, c, h, w)` | batch of video clips |
| `(n, c, d, h, w)` | batch of 3D volumes, medical imaging |

## Operation cheat sheet

| Task | Call |
|------|------|
| Per feature statistics | `X.mean(axis=0)`, `X.std(axis=0)` |
| Per sample statistics | `X.mean(axis=1)` |
| Keep the axis for broadcasting | `X.sum(axis=1, keepdims=True)` |
| Predicted class | `logits.argmax(axis=-1)` |
| Stable softmax | `x -= x.max(axis=-1, keepdims=True)` then exp and normalize |
| Add a batch axis | `x[None]` or `np.expand_dims(x, 0)` |
| Flatten for a dense head | `x.reshape(x.shape[0], -1)` |
| NCHW to NHWC | `np.transpose(x, (0, 2, 3, 1))` |
| Collate samples into a batch | `np.stack(samples, axis=0)` |
| Join feature blocks | `np.concatenate([a, b], axis=1)` |
| Top k, fast | `np.argpartition(-s, k)[:k]` |
| Filter rows | `X[mask]` where `mask` is boolean |
| Gather by index | `table[ids]` |
| Scatter add | `np.add.at(target, idx, values)` |
| One hot | `np.eye(C)[labels]` |
| Time series windows | `sliding_window_view(x, w)` |
| Pairwise distances | `q2 + d2 - 2 * Q @ D.T` |
| Guard a division | `a / (b + 1e-8)` |
| Check for NaN | `np.isfinite(x).all()` |
| Compare floats | `np.allclose(a, b)` |
| Own your data | `.copy()` |

## Where to go next

- **pandas** for labelled, heterogeneous tabular data. It is NumPy underneath.
- **scikit-learn** for classical models and the fit / transform pipeline pattern this notebook rebuilt by hand.
- **PyTorch** or **JAX** when you need autograd and a GPU. Both use the NumPy API almost verbatim.
- **Numba** or **Cython** when an algorithm genuinely cannot be vectorized.
- **Dask** or **Zarr** when arrays outgrow one machine.